
# NB_03 — Policies Incremental Bronze-to-Silver Load

## Purpose

This notebook incrementally processes insurance Policy data from the Bronze
Lakehouse into the Silver Lakehouse.

Unlike the Customer notebook, Policies uses `last_updated` as its watermark.

This allows the incremental process to detect both:

- Newly created policies
- Changes to existing policies

---

## Notebook at a Glance

### Processing Flow

`Bronze Policies`
→ `Read ETL Control`
→ `Read Watermark`
→ `Incremental Filter`
→ `Data Quality Validation`
→ `Deduplication`
→ `Transformation`
→ `Delta MERGE`
→ `Silver Policies`
→ `Audit Execution`
→ `Advance Watermark`

### Configuration

| Component | Value |
|---|---|
| Source | `LH_Bronze.dbo.bronze_policies` |
| Target | `LH_Silver.dbo.silver_policies` |
| Business Key | `policy_id` |
| Watermark | `last_updated` |
| Control Table | `LH_Silver.dbo.etl_control` |
| Audit Table | `LH_Silver.dbo.etl_batch_audit` |
| Load Type | `INCREMENTAL` |
| Storage | Delta |
| Write Pattern | Delta MERGE |

### Execution Outcomes

**SUCCESS**  
New or changed policies → Validate → Transform → MERGE → Audit → Advance watermark

**NO_DATA**  
No policies beyond the current watermark → Audit `NO_DATA` → Keep watermark unchanged

**FAILED**  
Processing error → Audit `FAILED` → Capture error → Keep watermark unchanged

> **Core design principle:** `last_updated` allows the Policy load to detect
> changes to existing records as well as newly created records.



## Step 1 — Initialize Policy Incremental Processing

This step initializes the Policy incremental processing execution.

Each execution receives a unique `batch_id` and captures its start timestamp
for operational auditing.

The Policy notebook uses the common ETL control and audit framework created
in `NB_01_ETL_Control_Framework`.

Unlike Customers, Policies uses `last_updated` as the incremental watermark,
allowing the process to detect both newly created policies and changes to
existing policies.

In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# ---------------------------------------------------------
# STEP 1 - INITIALIZE POLICY INCREMENTAL PROCESSING
# ---------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)
from delta.tables import DeltaTable
from datetime import datetime
import uuid

# Entity
SOURCE_NAME = "POLICIES"

# Framework tables
CONTROL_TABLE = "LH_Silver.dbo.etl_control"
AUDIT_TABLE   = "LH_Silver.dbo.etl_batch_audit"

# Orchestration identifier
PIPELINE_NAME = "PL_Insurance_Medallion_ETL"

# Execution identity
RUN_START_TIME = datetime.now()
BATCH_ID = str(uuid.uuid4())

print("Policy incremental processing initialized.")
print("---------------------------------------------")
print(f"Source name    : {SOURCE_NAME}")
print(f"Batch ID       : {BATCH_ID}")
print(f"Run start time : {RUN_START_TIME}")
print(f"Pipeline       : {PIPELINE_NAME}")


StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 3, Finished, Available, Finished, False)

Policy incremental processing initialized.
---------------------------------------------
Source name    : POLICIES
Batch ID       : 9268bbef-ebe8-4dc8-ab7d-da406cfb9911
Run start time : 2026-08-21 18:41:12.618419
Pipeline       : PL_Insurance_Medallion_ETL



## Step 2 — Read Policy Configuration from ETL Control

The Policy notebook retrieves its processing configuration from
`LH_Silver.dbo.etl_control`.

The control table determines:

- Bronze source table
- Silver target table
- Watermark column
- Current watermark
- Load type
- Active processing status

For Policies, the expected watermark column is `last_updated`.

Using metadata rather than hard-coded table configuration makes the
incremental framework reusable across multiple insurance entities.

In [2]:

# ---------------------------------------------------------
# STEP 2 - READ POLICY CONFIGURATION
# ---------------------------------------------------------

policy_config_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
)

config_count = policy_config_df.count()

if config_count != 1:
    raise ValueError(
        f"Expected exactly one active configuration for {SOURCE_NAME}, "
        f"but found {config_count}."
    )

display(policy_config_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b4d18edc-3f5c-432a-b35e-e9afb81cf83b)


## Step 3 — Load Policy Runtime Configuration

The active Policy metadata is converted into runtime variables used by the
remaining processing steps.

No physical table names or watermark values need to be hard-coded into the
processing logic.

For Policies, `last_updated` is especially important because it allows the
incremental process to detect both newly inserted policies and changes to
existing policies.

In [3]:

# ---------------------------------------------------------
# STEP 3 - LOAD POLICY RUNTIME CONFIGURATION
# ---------------------------------------------------------

config = policy_config_df.first()

SOURCE_TABLE     = config["source_table"]
TARGET_TABLE     = config["target_table"]
WATERMARK_COLUMN = config["watermark_column"]
LAST_WATERMARK   = config["last_watermark"]
LOAD_TYPE        = config["load_type"]

print("Policy runtime configuration loaded.")
print("---------------------------------------------")
print(f"Source name      : {SOURCE_NAME}")
print(f"Source table     : {SOURCE_TABLE}")
print(f"Target table     : {TARGET_TABLE}")
print(f"Watermark column : {WATERMARK_COLUMN}")
print(f"Last watermark   : {LAST_WATERMARK}")
print(f"Load type        : {LOAD_TYPE}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 5, Finished, Available, Finished, False)

Policy runtime configuration loaded.
---------------------------------------------
Source name      : POLICIES
Source table     : LH_Bronze.dbo.bronze_policies
Target table     : LH_Silver.dbo.silver_policies
Watermark column : last_updated
Last watermark   : 1900-01-01 00:00:00
Load type        : INCREMENTAL


In [4]:

# ---------------------------------------------------------
# STEP 4 - INSPECT BRONZE POLICY SOURCE
# ---------------------------------------------------------

bronze_policy_df = spark.table(SOURCE_TABLE)

bronze_policy_count = bronze_policy_df.count()

print("Bronze Policy source inspected.")
print("---------------------------------------------")
print(f"Source table : {SOURCE_TABLE}")
print(f"Bronze rows  : {bronze_policy_count}")
print()
print("Schema:")

bronze_policy_df.printSchema()

assert WATERMARK_COLUMN in bronze_policy_df.columns, \
    f"Watermark column '{WATERMARK_COLUMN}' not found in Bronze Policy table."

print()
print(f"Watermark column '{WATERMARK_COLUMN}' verified.")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 6, Finished, Available, Finished, False)

Bronze Policy source inspected.
---------------------------------------------
Source table : LH_Bronze.dbo.bronze_policies
Bronze rows  : 751

Schema:
root
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- policy_start_date: string (nullable = true)
 |-- policy_end_date: string (nullable = true)
 |-- annual_premium: string (nullable = true)
 |-- coverage_limit: string (nullable = true)
 |-- deductible: string (nullable = true)
 |-- policy_status: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- last_updated: string (nullable = true)


Watermark column 'last_updated' verified.


In [5]:
display(
    bronze_policy_df
    .orderBy(F.col(WATERMARK_COLUMN).desc())
    .limit(10)
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f53e6131-9b22-4bdb-bc45-88000dd75eee)


## Step 5 — Read Incremental Policy Records

The Policy source is filtered using the watermark stored in `etl_control`.

Only records where:

`last_updated > last_watermark`

are selected for processing.

Because `last_updated` changes when an existing Policy changes, this pattern
supports both:

- New Policy detection
- Existing Policy change detection

On the initial execution, the watermark is `1900-01-01`, so all existing
Bronze Policy records are expected to qualify.

In [6]:

# ---------------------------------------------------------
# STEP 5 - READ INCREMENTAL POLICY RECORDS
# ---------------------------------------------------------

bronze_policy_df = spark.table(SOURCE_TABLE)

incremental_policy_df = (
    bronze_policy_df
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(WATERMARK_COLUMN))
    )
    .filter(
        F.col("_watermark_ts") > F.lit(LAST_WATERMARK)
    )
)

source_total_count = bronze_policy_df.count()
incremental_count  = incremental_policy_df.count()

print("Bronze Policy incremental read completed.")
print("---------------------------------------------")
print(f"Source table        : {SOURCE_TABLE}")
print(f"Total Bronze rows   : {source_total_count}")
print(f"Last watermark      : {LAST_WATERMARK}")
print(f"Watermark column    : {WATERMARK_COLUMN}")
print(f"Incremental records : {incremental_count}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 8, Finished, Available, Finished, False)

Bronze Policy incremental read completed.
---------------------------------------------
Source table        : LH_Bronze.dbo.bronze_policies
Total Bronze rows   : 751
Last watermark      : 1900-01-01 00:00:00
Watermark column    : last_updated
Incremental records : 751


In [7]:
display(
    incremental_policy_df
    .orderBy(F.col("_watermark_ts").desc())
    .limit(10)
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, def0bc2b-07d4-4b6c-b754-5ab47c0a4e83)


## Step 6 — Validate Incremental Policy Data

Incremental Policy records are validated before they are allowed to enter
the Silver layer.

The validation rules protect key insurance business attributes including
policy identity, customer ownership, policy effective dates, financial
values, status, and incremental-processing timestamps.

Records failing validation are separated from valid records so that bad
source data does not contaminate the Silver layer.

Duplicate Policy records are handled separately during the deduplication step.

In [8]:
# ---------------------------------------------------------
# STEP 6 - POLICY DATA-QUALITY VALIDATION
# ---------------------------------------------------------

validated_policy_df = (
    incremental_policy_df

    # Standardize important data types
    .withColumn(
        "_parsed_start_date",
        F.to_date(F.col("policy_start_date"))
    )
    .withColumn(
        "_parsed_end_date",
        F.to_date(F.col("policy_end_date"))
    )
    .withColumn(
        "_parsed_last_updated",
        F.to_timestamp(F.col("last_updated"))
    )

    # Determine validation failure reason
    .withColumn(
        "reject_reason",

        F.when(
            F.col("policy_id").isNull() |
            (F.trim(F.col("policy_id")) == ""),
            "MISSING_POLICY_ID"
        )

        .when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "MISSING_CUSTOMER_ID"
        )

        .when(
            F.col("product_type").isNull() |
            (F.trim(F.col("product_type")) == ""),
            "MISSING_PRODUCT_TYPE"
        )

        .when(
            F.col("_parsed_start_date").isNull(),
            "INVALID_POLICY_START_DATE"
        )

        .when(
            F.col("_parsed_end_date").isNull(),
            "INVALID_POLICY_END_DATE"
        )

        .when(
            F.col("_parsed_end_date") < F.col("_parsed_start_date"),
            "END_DATE_BEFORE_START_DATE"
        )

        .when(
            F.col("annual_premium").isNull() |
            (F.col("annual_premium") <= 0),
            "INVALID_ANNUAL_PREMIUM"
        )

        .when(
            F.col("coverage_limit").isNull() |
            (F.col("coverage_limit") <= 0),
            "INVALID_COVERAGE_LIMIT"
        )

        .when(
            F.col("deductible").isNull() |
            (F.col("deductible") < 0),
            "INVALID_DEDUCTIBLE"
        )

        .when(
            ~F.col("policy_status").isin(
                "Active",
                "Pending",
                "Expired",
                "Cancelled"
            ),
            "INVALID_POLICY_STATUS"
        )

        .when(
            F.col("_parsed_last_updated").isNull(),
            "INVALID_LAST_UPDATED"
        )
    )
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 10, Finished, Available, Finished, False)

In [9]:

valid_policy_df = (
    validated_policy_df
    .filter(F.col("reject_reason").isNull())
)

rejected_policy_df = (
    validated_policy_df
    .filter(F.col("reject_reason").isNotNull())
)

valid_count  = valid_policy_df.count()
reject_count = rejected_policy_df.count()

print("Policy data-quality validation completed.")
print("---------------------------------------------")
print(f"Incremental records : {incremental_count}")
print(f"Valid records       : {valid_count}")
print(f"Rejected records    : {reject_count}")
print(f"Reconciliation      : {valid_count + reject_count}")

assert valid_count + reject_count == incremental_count, \
    "Policy validation reconciliation failed."

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 11, Finished, Available, Finished, False)

Policy data-quality validation completed.
---------------------------------------------
Incremental records : 751
Valid records       : 750
Rejected records    : 1
Reconciliation      : 751


In [10]:

display(
    rejected_policy_df
    .select(
        "policy_id",
        "customer_id",
        "product_type",
        "policy_start_date",
        "policy_end_date",
        "annual_premium",
        "coverage_limit",
        "deductible",
        "policy_status",
        "last_updated",
        "reject_reason"
    )
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a053c5f8-0458-4e76-8ba8-4a81ddd270aa)


## Step 7 — Deduplicate Incremental Policy Records

Policy records are uniquely identified by the business key:

`policy_id`

The Bronze layer may contain multiple versions of the same Policy.

Because Policies include a `last_updated` timestamp, the Silver processing
logic can deterministically retain the most recent version of each Policy.

For each `policy_id`:

- The record with the newest `last_updated` is retained.
- Older duplicate versions are excluded from Silver processing.

This prevents multiple source rows for the same Policy from being passed to
the Delta MERGE.

In [11]:

# ---------------------------------------------------------
# STEP 7 - DEDUPLICATE INCREMENTAL POLICY RECORDS
# ---------------------------------------------------------

from pyspark.sql.window import Window

policy_window = (
    Window
    .partitionBy("policy_id")
    .orderBy(
        F.col("_parsed_last_updated").desc()
    )
)

ranked_policy_df = (
    valid_policy_df
    .withColumn(
        "_policy_rank",
        F.row_number().over(policy_window)
    )
)

deduplicated_policy_df = (
    ranked_policy_df
    .filter(F.col("_policy_rank") == 1)
)

duplicate_policy_df = (
    ranked_policy_df
    .filter(F.col("_policy_rank") > 1)
)

deduplicated_count = deduplicated_policy_df.count()
duplicate_count    = duplicate_policy_df.count()

print("Policy deduplication completed.")
print("---------------------------------------------")
print(f"Valid records        : {valid_count}")
print(f"Unique Policy records: {deduplicated_count}")
print(f"Duplicate versions   : {duplicate_count}")
print(f"Reconciliation       : {deduplicated_count + duplicate_count}")

assert (
    deduplicated_count + duplicate_count
    == valid_count
), "Policy deduplication reconciliation failed."

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 13, Finished, Available, Finished, False)

Policy deduplication completed.
---------------------------------------------
Valid records        : 750
Unique Policy records: 749
Duplicate versions   : 1
Reconciliation       : 750


In [12]:
display(
    duplicate_policy_df
    .select(
        "policy_id",
        "customer_id",
        "product_type",
        "policy_status",
        "last_updated",
        "_policy_rank"
    )
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 038f1354-078d-4e5f-9b1a-1483bae370d6)


## Step 8 — Transform Policies for Silver

After validation and deduplication, the surviving Policy records are
transformed into the Silver representation.

This step:

- Applies explicit business data types
- Removes temporary processing columns
- Preserves the Policy business key
- Preserves `last_updated` for change tracking
- Adds a Silver processing timestamp

The resulting DataFrame represents the records eligible for Delta MERGE
into `silver_policies`.

In [35]:

# ---------------------------------------------------------
# STEP 8 - TRANSFORM POLICIES FOR SILVER
# ---------------------------------------------------------

silver_ready_policy_df = (
    deduplicated_policy_df
    .select(
        F.col("policy_id"),
        F.col("customer_id"),

        F.upper(F.trim(F.col("product_type")))
            .alias("product_type"),

        F.col("_parsed_start_date")
            .alias("policy_start_date"),

        F.col("_parsed_end_date")
            .alias("policy_end_date"),

        F.col("annual_premium")
            .cast("double")
            .alias("annual_premium"),

        F.col("coverage_limit")
            .cast("double")
            .alias("coverage_limit"),

        F.col("deductible")
            .cast("double")
            .alias("deductible"),

        F.upper(F.trim(F.col("policy_status")))
            .alias("policy_status"),

        F.col("agent_id"),

        F.col("_parsed_last_updated")
            .alias("last_updated"),

        F.current_timestamp()
            .alias("_silver_processed_ts")
    )
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 37, Finished, Available, Finished, False)

In [36]:

silver_ready_policy_df.printSchema()

display(
    silver_ready_policy_df
    .orderBy(F.col("last_updated").desc())
    .limit(10)
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 38, Finished, Available, Finished, False)

root
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- policy_start_date: date (nullable = true)
 |-- policy_end_date: date (nullable = true)
 |-- annual_premium: double (nullable = true)
 |-- coverage_limit: double (nullable = true)
 |-- deductible: double (nullable = true)
 |-- policy_status: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- _silver_processed_ts: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, 77e7f4be-fa4e-46de-9fce-ceca214e04da)


## Step 9 — Analyze Delta MERGE Impact

Before executing the Delta MERGE, incoming Silver-ready Policy records are
compared with the existing Silver Policy table.

Each incoming Policy is classified as:

- **INSERT** — `policy_id` does not exist in Silver.
- **UPDATE** — `policy_id` exists and one or more business attributes changed.
- **NO-OP** — `policy_id` exists and the business attributes are unchanged.

This pre-MERGE analysis provides reconciliation metrics and prevents
unchanged records from being treated as updates.

In [37]:

# ---------------------------------------------------------
# STEP 9A - INSPECT EXISTING SILVER POLICY TABLE
# ---------------------------------------------------------

existing_silver_policy_df = spark.table(TARGET_TABLE)

existing_silver_count = existing_silver_policy_df.count()

print("Existing Silver Policy table inspected.")
print("---------------------------------------------")
print(f"Target table         : {TARGET_TABLE}")
print(f"Existing Silver rows : {existing_silver_count}")

existing_silver_policy_df.printSchema()

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 39, Finished, Available, Finished, False)

Existing Silver Policy table inspected.
---------------------------------------------
Target table         : LH_Silver.dbo.silver_policies
Existing Silver rows : 750
root
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- policy_start_date: date (nullable = true)
 |-- policy_end_date: date (nullable = true)
 |-- annual_premium: double (nullable = true)
 |-- coverage_limit: double (nullable = true)
 |-- deductible: double (nullable = true)
 |-- policy_status: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- _silver_processed_ts: timestamp (nullable = true)



In [38]:

# ---------------------------------------------------------
# STEP 9B - IDENTIFY POLICY INSERT CANDIDATES
# ---------------------------------------------------------

insert_policy_df = (
    silver_ready_policy_df.alias("src")
    .join(
        existing_silver_policy_df.alias("tgt"),
        F.col("src.policy_id") == F.col("tgt.policy_id"),
        "left_anti"
    )
)

insert_count = insert_policy_df.count()

print(f"Policy INSERT candidates : {insert_count}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 40, Finished, Available, Finished, False)

Policy INSERT candidates : 0


In [39]:

matched_policy_df = (
    silver_ready_policy_df.alias("src")
    .join(
        existing_silver_policy_df.alias("tgt"),
        F.col("src.policy_id") == F.col("tgt.policy_id"),
        "inner"
    )
)

matched_count = matched_policy_df.count()

print(f"Matched Policies : {matched_count}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 41, Finished, Available, Finished, False)

Matched Policies : 749


In [49]:

# ---------------------------------------------------------
# STEP 9D - DETECT ACTUAL POLICY CHANGES
# ---------------------------------------------------------

change_condition = (
    ~F.col("src.customer_id").eqNullSafe(F.col("tgt.customer_id"))
    |
    ~F.col("src.product_type").eqNullSafe(F.col("tgt.product_type"))
    |
    ~F.col("src.policy_start_date").eqNullSafe(F.col("tgt.policy_start_date"))
    |
    ~F.col("src.policy_end_date").eqNullSafe(F.col("tgt.policy_end_date"))
    |
    ~F.col("src.annual_premium").eqNullSafe(F.col("tgt.annual_premium"))
    |
    ~F.col("src.coverage_limit").eqNullSafe(F.col("tgt.coverage_limit"))
    |
    ~F.col("src.deductible").eqNullSafe(F.col("tgt.deductible"))
    |
    ~F.col("src.policy_status").eqNullSafe(F.col("tgt.policy_status"))
    |
    ~F.col("src.agent_id").eqNullSafe(F.col("tgt.agent_id"))
)

##changed_policy_df = matched_policy_df.filter(change_condition)
##unchanged_policy_df = matched_policy_df.filter(~change_condition)
changed_policy_df = (
    matched_policy_df
    .filter(change_condition)
    .select("src.*")
)

unchanged_policy_df = (
    matched_policy_df
    .filter(~change_condition)
    .select("src.*")
)

update_count    = changed_policy_df.count()
unchanged_count = unchanged_policy_df.count()

print("Policy change detection completed.")
print("---------------------------------------------")
print(f"Silver-ready Policies : {silver_ready_count}")
print(f"INSERT candidates     : {insert_count}")
print(f"Matched Policies      : {matched_count}")
print(f"Actual UPDATEs        : {update_count}")
print(f"NO-OP Policies        : {unchanged_count}")
print(f"Reconciliation        : {insert_count + update_count + unchanged_count}")

assert insert_count + update_count + unchanged_count == silver_ready_count, \
    "Policy MERGE impact reconciliation failed."

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 51, Finished, Available, Finished, False)

Policy change detection completed.
---------------------------------------------
Silver-ready Policies : 749
INSERT candidates     : 0
Matched Policies      : 749
Actual UPDATEs        : 0
NO-OP Policies        : 749
Reconciliation        : 749


In [41]:

# ---------------------------------------------------------
# STEP 9E - DIAGNOSE POLICY CHANGE DETECTION
# ---------------------------------------------------------

comparison_df = (
    silver_ready_policy_df.alias("src")
    .join(
        existing_silver_policy_df.alias("tgt"),
        F.col("src.policy_id") == F.col("tgt.policy_id"),
        "inner"
    )
    .select(
        F.col("src.policy_id"),

        (~F.col("src.customer_id")
            .eqNullSafe(F.col("tgt.customer_id")))
            .alias("customer_changed"),

        (~F.col("src.product_type")
            .eqNullSafe(F.col("tgt.product_type")))
            .alias("product_changed"),

        (~F.col("src.policy_start_date")
            .eqNullSafe(F.col("tgt.policy_start_date")))
            .alias("start_date_changed"),

        (~F.col("src.policy_end_date")
            .eqNullSafe(F.col("tgt.policy_end_date")))
            .alias("end_date_changed"),

        (~F.col("src.annual_premium")
            .eqNullSafe(F.col("tgt.annual_premium")))
            .alias("premium_changed"),

        (~F.col("src.coverage_limit")
            .eqNullSafe(F.col("tgt.coverage_limit")))
            .alias("coverage_changed"),

        (~F.col("src.deductible")
            .eqNullSafe(F.col("tgt.deductible")))
            .alias("deductible_changed"),

        (~F.col("src.policy_status")
            .eqNullSafe(F.col("tgt.policy_status")))
            .alias("status_changed"),

        (~F.col("src.agent_id")
            .eqNullSafe(F.col("tgt.agent_id")))
            .alias("agent_changed")
    )
)

display(comparison_df.limit(20))


StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 781f3325-1133-492f-8570-2a57e5830e63)

In [42]:

# ---------------------------------------------------------
# STEP 9F - INSPECT PRODUCT TYPE AND STATUS DIFFERENCES
# ---------------------------------------------------------

value_comparison_df = (
    silver_ready_policy_df.alias("src")
    .join(
        existing_silver_policy_df.alias("tgt"),
        F.col("src.policy_id") == F.col("tgt.policy_id"),
        "inner"
    )
    .select(
        F.col("src.policy_id"),

        F.col("src.product_type")
            .alias("bronze_product_type"),

        F.col("tgt.product_type")
            .alias("silver_product_type"),

        F.col("src.policy_status")
            .alias("bronze_policy_status"),

        F.col("tgt.policy_status")
            .alias("silver_policy_status"),

        F.col("src.last_updated")
            .alias("bronze_last_updated")
    )
)

display(value_comparison_df.limit(20))

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c40fdd1-5588-4f3b-8f60-6627000dbf4a)


## Step 10 — Execute the Policy Delta MERGE

The incoming Policy records have already been classified into three categories:

- **INSERT** — Policy does not exist in Silver.
- **UPDATE** — Policy exists and one or more business attributes changed.
- **NO-OP** — Policy exists and no business attributes changed.

Only INSERT and UPDATE records are sent to the Delta MERGE.

NO-OP records are intentionally excluded so that unchanged Silver rows are not
rewritten unnecessarily.

### Merge Key

The Policy business key is:

`policy_id`

### Merge Behavior

If `policy_id` already exists in Silver and the record has changed:

`UPDATE`

If `policy_id` does not exist in Silver:

`INSERT`

If there are no INSERT or UPDATE candidates:

`MERGE is skipped`

This reduces unnecessary Delta writes and keeps incremental processing efficient.

In [50]:

# ---------------------------------------------------------
# STEP 10 - MERGE POLICY CHANGES INTO SILVER
# ---------------------------------------------------------

merge_columns = [
    "policy_id",
    "customer_id",
    "product_type",
    "policy_start_date",
    "policy_end_date",
    "annual_premium",
    "coverage_limit",
    "deductible",
    "policy_status",
    "agent_id",
    "last_updated",
    "_silver_processed_ts"
]

policy_insert_merge_df = (
    insert_policy_df
    .select(*merge_columns)
)

policy_update_merge_df = (
    changed_policy_df
    .select(*merge_columns)
)

policy_merge_source_df = (
    policy_insert_merge_df
    .unionByName(policy_update_merge_df)
)

merge_source_count = policy_merge_source_df.count()

print("Preparing Policy Delta MERGE.")
print("----------------------------------------")
print(f"Insert records : {insert_count}")
print(f"Changed records: {update_count}")
print(f"MERGE source   : {merge_source_count}")

if merge_source_count > 0:

    silver_policy_delta = DeltaTable.forName(
        spark,
        TARGET_TABLE
    )

    (
        silver_policy_delta.alias("tgt")
        .merge(
            policy_merge_source_df.alias("src"),
            "tgt.policy_id = src.policy_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Policy Delta MERGE completed.")

else:

    print("No Policy inserts or updates detected.")
    print("Delta MERGE skipped.")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 52, Finished, Available, Finished, False)

Preparing Policy Delta MERGE.
----------------------------------------
Insert records : 0
Changed records: 0
MERGE source   : 0
No Policy inserts or updates detected.
Delta MERGE skipped.



## Step 11 — Write the Policy Batch Audit Record

The Policy processing cycle completed successfully.

For this execution:

- Incremental source records were read from Bronze.
- Invalid records were rejected during validation.
- Older duplicate Policy versions were excluded during deduplication.
- No INSERT or UPDATE was required because all surviving Policy records
  already matched the current Silver representation.

The execution result is now written to:

`LH_Silver.dbo.etl_batch_audit`

This provides operational traceability for the Policy incremental load.

In [51]:

audit_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), True),
    StructField("source_count", LongType(), False),
    StructField("insert_count", LongType(), False),
    StructField("update_count", LongType(), False),
    StructField("reject_count", LongType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 53, Finished, Available, Finished, False)

In [52]:
# ---------------------------------------------------------
# STEP 11A - BUILD POLICY AUDIT RECORD
# ---------------------------------------------------------

RUN_END_TIME = datetime.now()

# Validation rejects + duplicate versions excluded
total_reject_count = reject_count + duplicate_count

policy_audit_record = [
    (
        BATCH_ID,
        PIPELINE_NAME,
        SOURCE_NAME,
        RUN_START_TIME,
        RUN_END_TIME,
        incremental_count,
        insert_count,
        update_count,
        total_reject_count,
        "SUCCESS",
        None
    )
]

policy_audit_df = spark.createDataFrame(
    policy_audit_record,
    schema=audit_schema
)

display(policy_audit_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9489a70f-a7f4-49c9-bd88-99c8a728dbac)

In [53]:
# ---------------------------------------------------------
# STEP 11B - PERSIST POLICY AUDIT RECORD
# ---------------------------------------------------------

(
    policy_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

print("Policy audit record written successfully.")
print(f"Batch ID : {BATCH_ID}")
print(f"Table    : {SOURCE_NAME}")
print(f"Status   : SUCCESS")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 55, Finished, Available, Finished, False)

Policy audit record written successfully.
Batch ID : 9268bbef-ebe8-4dc8-ab7d-da406cfb9911
Table    : POLICIES
Status   : SUCCESS


In [54]:

# ---------------------------------------------------------
# STEP 11C - VERIFY POLICY AUDIT RECORD
# ---------------------------------------------------------

policy_audit_check_df = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)

display(policy_audit_check_df)

audit_row_count = policy_audit_check_df.count()

assert audit_row_count == 1, \
    f"Expected exactly one Policy audit row, found {audit_row_count}."

print("Policy audit persistence verified.")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ef8e67e6-12dc-4848-9cdb-d67e9f441171)

Policy audit persistence verified.



## Step 12 — Advance the Policy Watermark

After successful Policy processing, advance the control-table watermark
to the maximum `last_updated` value observed in the successfully processed
incremental batch.

The watermark is updated only after successful processing and audit completion.

This ensures:

- already processed Policy records are not reread
- incremental execution is restartable
- failed executions do not incorrectly advance the watermark
- the next run processes only newer Policy changes

In [55]:

# ---------------------------------------------------------
# STEP 12A - CALCULATE NEW POLICY WATERMARK
# ---------------------------------------------------------

new_policy_watermark = (
    incremental_policy_df
    .agg(
        F.max(F.col("_watermark_ts"))
        .alias("new_watermark")
    )
    .first()["new_watermark"]
)

print("Policy watermark calculated.")
print("----------------------------------------")
print(f"Previous watermark : {LAST_WATERMARK}")
print(f"New watermark      : {new_policy_watermark}")

assert new_policy_watermark is not None, \
    "Policy watermark calculation returned NULL."

assert new_policy_watermark >= LAST_WATERMARK, \
    "New Policy watermark cannot be earlier than previous watermark."

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 57, Finished, Available, Finished, False)

Policy watermark calculated.
----------------------------------------
Previous watermark : 1900-01-01 00:00:00
New watermark      : 2027-12-08 00:00:00


In [56]:
# ---------------------------------------------------------
# STEP 12B - UPDATE POLICY WATERMARK
# ---------------------------------------------------------

control_delta = DeltaTable.forName(
    spark,
    CONTROL_TABLE
)

(
    control_delta.alias("tgt")
    .update(
        condition=(
            (F.col("source_name") == SOURCE_NAME) &
            (F.col("is_active") == True)
        ),
        set={
            "last_watermark": F.lit(new_policy_watermark),
            "_updated_ts": F.current_timestamp()
        }
    )
)

print("Policy watermark updated successfully.")
print("----------------------------------------")
print(f"Source name   : {SOURCE_NAME}")
print(f"Old watermark: {LAST_WATERMARK}")
print(f"New watermark: {new_policy_watermark}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 58, Finished, Available, Finished, False)

Policy watermark updated successfully.
----------------------------------------
Source name   : POLICIES
Old watermark: 1900-01-01 00:00:00
New watermark: 2027-12-08 00:00:00


In [57]:
# ---------------------------------------------------------
# STEP 12C - VERIFY UPDATED POLICY WATERMARK
# ---------------------------------------------------------

updated_policy_control_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
    .select(
        "source_name",
        "watermark_column",
        "last_watermark",
        "load_type",
        "is_active",
        "_updated_ts"
    )
)

display(updated_policy_control_df)

updated_policy_control = updated_policy_control_df.first()

assert updated_policy_control["last_watermark"] == new_policy_watermark, \
    "Policy watermark was not updated correctly."

print("Policy watermark verification PASSED.")
print(f"Stored watermark: {updated_policy_control['last_watermark']}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a2d468a5-d89f-46a2-a4cd-2f5599633863)

Policy watermark verification PASSED.
Stored watermark: 2027-12-08 00:00:00



## Step 13 — Validate Policy Incremental Restart Behavior

The Policy watermark has been advanced to the maximum successfully processed
`last_updated` value:

`2027-12-08 00:00:00`

The incremental filter is now executed again without changing Bronze data.

Because the process selects only records where:

`last_updated > stored watermark`

the expected result is:

**0 incremental Policy records**

This validates that already processed Policy records are not reprocessed.

In [58]:

# ---------------------------------------------------------
# STEP 13 - VERIFY POLICY RESTART BEHAVIOR
# ---------------------------------------------------------

restart_config = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .first()
)

RESTART_WATERMARK = restart_config["last_watermark"]

restart_bronze_policy_df = spark.table(SOURCE_TABLE)

restart_incremental_policy_df = (
    restart_bronze_policy_df
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(WATERMARK_COLUMN))
    )
    .filter(
        F.col("_watermark_ts") > F.lit(RESTART_WATERMARK)
    )
)

restart_incremental_count = restart_incremental_policy_df.count()

print("Policy restart test completed.")
print("----------------------------------------")
print(f"Bronze Policy rows   : {restart_bronze_policy_df.count()}")
print(f"Stored watermark     : {RESTART_WATERMARK}")
print(f"Incremental records  : {restart_incremental_count}")

assert restart_incremental_count == 0, \
    "Expected zero Policy records after watermark advancement."

print()
print("Policy incremental restart PASSED.")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 60, Finished, Available, Finished, False)

Policy restart test completed.
----------------------------------------
Bronze Policy rows   : 751
Stored watermark     : 2027-12-08 00:00:00
Incremental records  : 0

Policy incremental restart PASSED.



## Step 14 — Controlled Policy UPDATE Test

The initial Policy incremental load and restart test have completed successfully.

We will now simulate a real source-system Policy change.

The test will:

1. Select an existing Policy from Bronze.
2. Create a newer version of that Policy.
3. Change its annual premium.
4. Set `last_updated` greater than the current watermark.
5. Append the new version to Bronze.
6. Verify that incremental processing detects exactly one record.
7. Validate and deduplicate the record.
8. Detect it as an UPDATE.
9. MERGE the change into Silver.
10. Advance the Policy watermark.

Expected result:

`1 incremental record → 1 UPDATE → 0 INSERT`

In [59]:
# ---------------------------------------------------------
# STEP 14A - SELECT POLICY FOR CONTROLLED UPDATE TEST
# ---------------------------------------------------------

TEST_POLICY_ID = "POL000001"

original_test_policy_df = (
    spark.table(SOURCE_TABLE)
    .filter(F.col("policy_id") == TEST_POLICY_ID)
    .orderBy(F.col("last_updated").desc())
    .limit(1)
)

test_policy_count = original_test_policy_df.count()

assert test_policy_count == 1, \
    f"Expected Policy {TEST_POLICY_ID} to exist."

print("Existing Policy selected for UPDATE test.")
print("----------------------------------------")
print(f"Policy ID : {TEST_POLICY_ID}")

display(original_test_policy_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 61, Finished, Available, Finished, False)

Existing Policy selected for UPDATE test.
----------------------------------------
Policy ID : POL000001


SynapseWidget(Synapse.DataFrame, 559103d6-66ec-46f4-aea7-15c56edc5784)

In [62]:
# ---------------------------------------------------------
# STEP 14B - CREATE UPDATED POLICY VERSION
# PRESERVE ORIGINAL BRONZE DATA TYPES
# ---------------------------------------------------------

TEST_NEW_PREMIUM = "527.01"
TEST_NEW_LAST_UPDATED = "2027-12-09"

updated_test_policy_df = (
    original_test_policy_df
    .withColumn(
        "annual_premium",
        F.lit(TEST_NEW_PREMIUM).cast("string")
    )
    .withColumn(
        "last_updated",
        F.lit(TEST_NEW_LAST_UPDATED).cast("string")
    )
)

print("Updated Policy test version created.")
print("----------------------------------------")
print(f"Policy ID        : {TEST_POLICY_ID}")
print(f"New premium      : {TEST_NEW_PREMIUM}")
print(f"New last_updated : {TEST_NEW_LAST_UPDATED}")

updated_test_policy_df.printSchema()

display(updated_test_policy_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 64, Finished, Available, Finished, False)

Updated Policy test version created.
----------------------------------------
Policy ID        : POL000001
New premium      : 527.01
New last_updated : 2027-12-09
root
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- policy_start_date: string (nullable = true)
 |-- policy_end_date: string (nullable = true)
 |-- annual_premium: string (nullable = false)
 |-- coverage_limit: string (nullable = true)
 |-- deductible: string (nullable = true)
 |-- policy_status: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- last_updated: string (nullable = false)



SynapseWidget(Synapse.DataFrame, 82712573-26a7-4f8c-a862-dfbb2e7828da)

In [63]:

# ---------------------------------------------------------
# STEP 14C - APPEND UPDATED POLICY VERSION TO BRONZE
# ---------------------------------------------------------

bronze_count_before_update = spark.table(SOURCE_TABLE).count()

(
    updated_test_policy_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(SOURCE_TABLE)
)

bronze_count_after_update = spark.table(SOURCE_TABLE).count()

print("Updated Policy version appended to Bronze.")
print("----------------------------------------")
print(f"Policy ID          : {TEST_POLICY_ID}")
print(f"Bronze rows before : {bronze_count_before_update}")
print(f"Bronze rows after  : {bronze_count_after_update}")
print(f"Net increase       : {bronze_count_after_update - bronze_count_before_update}")

assert bronze_count_after_update == bronze_count_before_update + 1, \
    "Expected Bronze Policy table to increase by exactly one row."

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 65, Finished, Available, Finished, False)

Updated Policy version appended to Bronze.
----------------------------------------
Policy ID          : POL000001
Bronze rows before : 751
Bronze rows after  : 752
Net increase       : 1


In [64]:

# ---------------------------------------------------------
# STEP 14D - VERIFY POLICY VERSIONS IN BRONZE
# ---------------------------------------------------------

test_policy_versions_df = (
    spark.table(SOURCE_TABLE)
    .filter(F.col("policy_id") == TEST_POLICY_ID)
    .orderBy(F.col("last_updated").desc())
)

display(
    test_policy_versions_df.select(
        "policy_id",
        "customer_id",
        "product_type",
        "annual_premium",
        "policy_status",
        "last_updated"
    )
)

test_policy_version_count = test_policy_versions_df.count()

print("----------------------------------------")
print(f"Policy ID       : {TEST_POLICY_ID}")
print(f"Bronze versions : {test_policy_version_count}")

assert test_policy_version_count == 2, \
    f"Expected exactly 2 Bronze versions, found {test_policy_version_count}."

print("Policy Bronze version verification PASSED.")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 66, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 77b35835-acbb-482d-bc5e-11619fe28173)

----------------------------------------
Policy ID       : POL000001
Bronze versions : 2
Policy Bronze version verification PASSED.



## Step 15 — Validate Incremental Detection of an Updated Policy

A controlled update was added to Bronze for an existing Policy:

- **Policy ID:** `POL000001`
- **Previous annual premium:** `427.01`
- **Updated annual premium:** `527.01`
- **Previous Bronze version:** `2022-12-12`
- **New `last_updated`:** `2027-12-09`
- **Current stored watermark:** `2027-12-08`

The purpose of this step is to verify that the watermark-based incremental
processing correctly detects only the newly updated Policy version.

The pipeline will:

1. Read the current Policy watermark from the control table.
2. Read the Bronze Policy table.
3. Convert `last_updated` into the watermark timestamp.
4. Select only records where:

   `last_updated > stored watermark`

5. Verify that exactly **one** incremental Policy record is detected.
6. Confirm that the older version of the same Policy is not reprocessed.

### Expected Result

`POL000001` with annual premium `527.01` should be the only incremental record.

Expected incremental count:

**1**

This validates that the Policy pipeline performs true watermark-based
incremental processing rather than rereading the complete Bronze dataset.

The detected record will next be validated, transformed, and compared with
the existing Silver Policy to determine whether it represents an INSERT,
UPDATE, or NO-OP.


In [65]:

# ---------------------------------------------------------
# STEP 15A - READ CURRENT POLICY WATERMARK
# ---------------------------------------------------------

update_test_config = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
    .first()
)

UPDATE_TEST_WATERMARK = update_test_config["last_watermark"]
UPDATE_TEST_WATERMARK_COLUMN = update_test_config["watermark_column"]

print("Policy UPDATE-test configuration loaded.")
print("----------------------------------------")
print(f"Source name      : {SOURCE_NAME}")
print(f"Watermark column : {UPDATE_TEST_WATERMARK_COLUMN}")
print(f"Stored watermark : {UPDATE_TEST_WATERMARK}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 67, Finished, Available, Finished, False)

Policy UPDATE-test configuration loaded.
----------------------------------------
Source name      : POLICIES
Watermark column : last_updated
Stored watermark : 2027-12-08 00:00:00


In [66]:

# ---------------------------------------------------------
# STEP 15B - DETECT NEW INCREMENTAL POLICY
# ---------------------------------------------------------

update_test_bronze_df = spark.table(SOURCE_TABLE)

update_test_incremental_df = (
    update_test_bronze_df
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(
            F.col(UPDATE_TEST_WATERMARK_COLUMN)
        )
    )
    .filter(
        F.col("_watermark_ts") > F.lit(UPDATE_TEST_WATERMARK)
    )
)

update_test_incremental_count = update_test_incremental_df.count()

print("Policy UPDATE-test incremental detection completed.")
print("----------------------------------------")
print(f"Bronze rows         : {update_test_bronze_df.count()}")
print(f"Stored watermark    : {UPDATE_TEST_WATERMARK}")
print(f"Incremental records : {update_test_incremental_count}")

assert update_test_incremental_count == 1, \
    f"Expected exactly 1 incremental Policy, found {update_test_incremental_count}."

display(
    update_test_incremental_df.select(
        "policy_id",
        "customer_id",
        "product_type",
        "annual_premium",
        "policy_status",
        "last_updated",
        "_watermark_ts"
    )
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 68, Finished, Available, Finished, False)

Policy UPDATE-test incremental detection completed.
----------------------------------------
Bronze rows         : 752
Stored watermark    : 2027-12-08 00:00:00
Incremental records : 1


SynapseWidget(Synapse.DataFrame, 1c0fdf0a-58df-44a2-8770-3ff288997f48)


## Step 16 — Validate and Transform the Incremental Policy

The watermark-based incremental read detected exactly one new Policy version.

Before comparing this record with Silver, the incremental Policy must pass
the same data-quality and transformation rules used by the main Policy load.

This step will:

1. Validate required Policy fields.
2. Parse Policy start and end dates.
3. Parse the `last_updated` timestamp.
4. Validate that the Policy end date is not earlier than the start date.
5. Cast numeric fields to their Silver data types.
6. Standardize categorical values such as product type and Policy status.
7. Produce a Silver-ready representation of the incremental Policy.

### Expected Result

The controlled test record `POL000001` should pass validation.

Expected:

- Incremental records: **1**
- Valid records: **1**
- Rejected records: **0**
- Silver-ready records: **1**

The Silver-ready Policy will then be compared against the existing Silver
version to determine whether the record represents an INSERT, UPDATE, or NO-OP.


In [67]:

# ---------------------------------------------------------
# STEP 16A - VALIDATE INCREMENTAL POLICY
# ---------------------------------------------------------

incremental_validated_df = (
    update_test_incremental_df

    .withColumn(
        "_parsed_start_date",
        F.to_date(F.col("policy_start_date"))
    )

    .withColumn(
        "_parsed_end_date",
        F.to_date(F.col("policy_end_date"))
    )

    .withColumn(
        "_parsed_last_updated",
        F.to_timestamp(F.col("last_updated"))
    )
)

incremental_rejected_df = (
    incremental_validated_df
    .filter(
        F.col("policy_id").isNull()
        | F.col("customer_id").isNull()
        | F.col("_parsed_start_date").isNull()
        | F.col("_parsed_end_date").isNull()
        | F.col("_parsed_last_updated").isNull()
        | (
            F.col("_parsed_end_date")
            < F.col("_parsed_start_date")
        )
    )
)

incremental_valid_df = (
    incremental_validated_df
    .filter(
        F.col("policy_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("_parsed_start_date").isNotNull()
        & F.col("_parsed_end_date").isNotNull()
        & F.col("_parsed_last_updated").isNotNull()
        & (
            F.col("_parsed_end_date")
            >= F.col("_parsed_start_date")
        )
    )
)

incremental_valid_count = incremental_valid_df.count()
incremental_reject_count = incremental_rejected_df.count()

print("Incremental Policy validation completed.")
print("----------------------------------------")
print(f"Incremental records : {update_test_incremental_count}")
print(f"Valid records       : {incremental_valid_count}")
print(f"Rejected records    : {incremental_reject_count}")
print(
    f"Reconciliation      : "
    f"{incremental_valid_count + incremental_reject_count}"
)

assert incremental_valid_count == 1, \
    f"Expected 1 valid Policy, found {incremental_valid_count}."

assert incremental_reject_count == 0, \
    f"Expected 0 rejected Policies, found {incremental_reject_count}."

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 69, Finished, Available, Finished, False)

Incremental Policy validation completed.
----------------------------------------
Incremental records : 1
Valid records       : 1
Rejected records    : 0
Reconciliation      : 1


In [68]:

# ---------------------------------------------------------
# STEP 16B - TRANSFORM INCREMENTAL POLICY FOR SILVER
# ---------------------------------------------------------

incremental_silver_ready_df = (
    incremental_valid_df
    .select(
        F.col("policy_id"),
        F.col("customer_id"),

        # Normalize categorical values
        F.upper(F.trim(F.col("product_type")))
            .alias("product_type"),

        F.col("_parsed_start_date")
            .alias("policy_start_date"),

        F.col("_parsed_end_date")
            .alias("policy_end_date"),

        F.col("annual_premium")
            .cast("double")
            .alias("annual_premium"),

        F.col("coverage_limit")
            .cast("double")
            .alias("coverage_limit"),

        F.col("deductible")
            .cast("double")
            .alias("deductible"),

        F.upper(F.trim(F.col("policy_status")))
            .alias("policy_status"),

        F.col("agent_id"),

        F.col("_parsed_last_updated")
            .alias("last_updated"),

        F.current_timestamp()
            .alias("_silver_processed_ts")
    )
)

incremental_silver_ready_count = incremental_silver_ready_df.count()

print("Incremental Policy Silver transformation completed.")
print("----------------------------------------")
print(f"Valid records       : {incremental_valid_count}")
print(f"Silver-ready records: {incremental_silver_ready_count}")

assert incremental_silver_ready_count == incremental_valid_count, \
    "Incremental Policy Silver transformation reconciliation failed."

display(incremental_silver_ready_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 70, Finished, Available, Finished, False)

Incremental Policy Silver transformation completed.
----------------------------------------
Valid records       : 1
Silver-ready records: 1


SynapseWidget(Synapse.DataFrame, 7cc0e5b4-65a2-445b-8faa-821fa91338d2)

## Step 17 — Detect Policy UPDATE Against Silver

The incremental Policy record has passed validation and has been transformed
into the canonical Silver representation.

This step compares the incoming Policy with the existing Silver record using
`policy_id` as the business key.

The comparison determines whether the incoming record represents:

- **INSERT** — Policy does not currently exist in Silver.
- **UPDATE** — Policy exists, but one or more business attributes changed.
- **NO-OP** — Policy exists and the business attributes are unchanged.

For this controlled test, `POL000001` already exists in Silver.

The annual premium was intentionally changed:

**427.01 → 527.01**

Therefore the expected result is:

- INSERT candidates: **0**
- UPDATE candidates: **1**
- NO-OP records: **0**

The `last_updated` and `_silver_processed_ts` columns are not used as business
change indicators. `last_updated` controls incremental ingestion, while
`_silver_processed_ts` records when Silver processing occurred.


In [69]:

# ---------------------------------------------------------
# STEP 17A - COMPARE INCREMENTAL POLICY WITH SILVER
# ---------------------------------------------------------

current_silver_policy_df = spark.table(TARGET_TABLE)

incoming = incremental_silver_ready_df.alias("src")
existing = current_silver_policy_df.alias("tgt")

policy_comparison_df = (
    incoming
    .join(
        existing,
        F.col("src.policy_id") == F.col("tgt.policy_id"),
        "left"
    )
)

# Existing vs new Policy
insert_condition = F.col("tgt.policy_id").isNull()

change_condition = (
    ~F.col("src.customer_id")
        .eqNullSafe(F.col("tgt.customer_id"))

    | ~F.col("src.product_type")
        .eqNullSafe(F.col("tgt.product_type"))

    | ~F.col("src.policy_start_date")
        .eqNullSafe(F.col("tgt.policy_start_date"))

    | ~F.col("src.policy_end_date")
        .eqNullSafe(F.col("tgt.policy_end_date"))

    | ~F.col("src.annual_premium")
        .eqNullSafe(F.col("tgt.annual_premium"))

    | ~F.col("src.coverage_limit")
        .eqNullSafe(F.col("tgt.coverage_limit"))

    | ~F.col("src.deductible")
        .eqNullSafe(F.col("tgt.deductible"))

    | ~F.col("src.policy_status")
        .eqNullSafe(F.col("tgt.policy_status"))

    | ~F.col("src.agent_id")
        .eqNullSafe(F.col("tgt.agent_id"))
)

insert_update_test_df = (
    policy_comparison_df
    .filter(insert_condition)
    .select("src.*")
)

changed_update_test_df = (
    policy_comparison_df
    .filter(
        F.col("tgt.policy_id").isNotNull()
        & change_condition
    )
    .select("src.*")
)

unchanged_update_test_df = (
    policy_comparison_df
    .filter(
        F.col("tgt.policy_id").isNotNull()
        & ~change_condition
    )
    .select("src.*")
)

test_insert_count = insert_update_test_df.count()
test_update_count = changed_update_test_df.count()
test_noop_count = unchanged_update_test_df.count()

print("Incremental Policy change detection completed.")
print("----------------------------------------")
print(f"Incoming Policies : {incremental_silver_ready_count}")
print(f"INSERT candidates : {test_insert_count}")
print(f"UPDATE candidates : {test_update_count}")
print(f"NO-OP Policies    : {test_noop_count}")
print(
    f"Reconciliation    : "
    f"{test_insert_count + test_update_count + test_noop_count}"
)

assert test_insert_count == 0, \
    f"Expected 0 INSERTs, found {test_insert_count}."

assert test_update_count == 1, \
    f"Expected exactly 1 UPDATE, found {test_update_count}."

assert test_noop_count == 0, \
    f"Expected 0 NO-OP records, found {test_noop_count}."

assert (
    test_insert_count
    + test_update_count
    + test_noop_count
) == incremental_silver_ready_count, \
    "Incremental Policy reconciliation failed."

print()
print("Controlled Policy UPDATE detection PASSED.")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 71, Finished, Available, Finished, False)

Incremental Policy change detection completed.
----------------------------------------
Incoming Policies : 1
INSERT candidates : 0
UPDATE candidates : 1
NO-OP Policies    : 0
Reconciliation    : 1

Controlled Policy UPDATE detection PASSED.


In [70]:

# ---------------------------------------------------------
# STEP 17B - INSPECT POLICY UPDATE DIFFERENCES
# ---------------------------------------------------------

policy_update_diff_df = (
    policy_comparison_df
    .filter(
        F.col("tgt.policy_id").isNotNull()
        & change_condition
    )
    .select(
        F.col("src.policy_id").alias("policy_id"),

        F.col("tgt.annual_premium")
            .alias("silver_old_premium"),

        F.col("src.annual_premium")
            .alias("bronze_new_premium"),

        F.col("tgt.product_type")
            .alias("silver_product_type"),

        F.col("src.product_type")
            .alias("bronze_product_type"),

        F.col("tgt.policy_status")
            .alias("silver_policy_status"),

        F.col("src.policy_status")
            .alias("bronze_policy_status"),

        F.col("tgt.last_updated")
            .alias("silver_last_updated"),

        F.col("src.last_updated")
            .alias("bronze_last_updated"),

        (
            ~F.col("src.annual_premium")
                .eqNullSafe(F.col("tgt.annual_premium"))
        ).alias("premium_changed"),

        (
            ~F.col("src.product_type")
                .eqNullSafe(F.col("tgt.product_type"))
        ).alias("product_type_changed"),

        (
            ~F.col("src.policy_status")
                .eqNullSafe(F.col("tgt.policy_status"))
        ).alias("status_changed")
    )
)

display(policy_update_diff_df)

diff_count = policy_update_diff_df.count()

assert diff_count == 1, \
    f"Expected exactly 1 changed Policy, found {diff_count}."

print()
print("Policy UPDATE difference inspection completed.")
print(f"Changed Policies : {diff_count}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 72, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dfcb4c43-4f32-434c-8b5c-6fd7016c2c2c)


Policy UPDATE difference inspection completed.
Changed Policies : 1



## Step 18 — MERGE Updated Policy into Silver

The controlled incremental test successfully identified one genuine Policy
UPDATE.

For `POL000001`, the annual premium changed:

**427.01 → 527.01**

The normalized product type and Policy status remain unchanged, confirming
that the UPDATE was caused by an actual business-data change rather than
formatting differences.

This step performs a Delta Lake MERGE using `policy_id` as the business key.

The MERGE will:

1. Match the incoming Policy to the existing Silver Policy using `policy_id`.
2. Update the existing Silver row with the latest business values.
3. Preserve one Silver row per Policy.
4. Update the source `last_updated` timestamp.
5. Record a new `_silver_processed_ts`.

### Expected Result

After the MERGE:

- `POL000001` occurs exactly **once** in Silver.
- `annual_premium` = **527.01**
- `product_type` = **RENTERS**
- `policy_status` = **CANCELLED**
- `last_updated` = **2027-12-09**
- Total Silver row count does **not increase**.

This demonstrates an in-place Delta UPDATE rather than inserting a duplicate
Policy.


In [72]:

# ---------------------------------------------------------
# STEP 18A - CAPTURE SILVER STATE BEFORE UPDATE MERGE
# ---------------------------------------------------------

silver_before_update_df = spark.table(TARGET_TABLE)

silver_count_before_update = silver_before_update_df.count()

test_policy_before_merge_df = (
    silver_before_update_df
    .filter(F.col("policy_id") == TEST_POLICY_ID)
)

test_policy_before_merge_count = test_policy_before_merge_df.count()

print("Silver pre-MERGE state captured.")
print("----------------------------------------")
print(f"Silver rows       : {silver_count_before_update}")
print(f"Test Policy rows  : {test_policy_before_merge_count}")

assert test_policy_before_merge_count == 1, \
    f"Expected exactly one {TEST_POLICY_ID} in Silver before MERGE."

display(
    test_policy_before_merge_df.select(
        "policy_id",
        "product_type",
        "annual_premium",
        "policy_status",
        "last_updated",
        "_silver_processed_ts"
    )
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 74, Finished, Available, Finished, False)

Silver pre-MERGE state captured.
----------------------------------------
Silver rows       : 750
Test Policy rows  : 1


SynapseWidget(Synapse.DataFrame, 10f47fef-c3b0-4afb-b746-b0c83f4e014e)

In [73]:
# ---------------------------------------------------------
# STEP 18B - MERGE UPDATED POLICY INTO SILVER
# ---------------------------------------------------------

silver_policy_delta = DeltaTable.forName(
    spark,
    TARGET_TABLE
)

(
    silver_policy_delta.alias("tgt")
    .merge(
        changed_update_test_df.alias("src"),
        "tgt.policy_id = src.policy_id"
    )
    .whenMatchedUpdate(
        set={
            "customer_id": "src.customer_id",
            "product_type": "src.product_type",
            "policy_start_date": "src.policy_start_date",
            "policy_end_date": "src.policy_end_date",
            "annual_premium": "src.annual_premium",
            "coverage_limit": "src.coverage_limit",
            "deductible": "src.deductible",
            "policy_status": "src.policy_status",
            "agent_id": "src.agent_id",
            "last_updated": "src.last_updated",
            "_silver_processed_ts": "src._silver_processed_ts"
        }
    )
    .execute()
)

print("Controlled Policy Delta UPDATE MERGE completed.")
print("----------------------------------------")
print(f"Policy ID       : {TEST_POLICY_ID}")
print(f"UPDATE records  : {test_update_count}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 75, Finished, Available, Finished, False)

Controlled Policy Delta UPDATE MERGE completed.
----------------------------------------
Policy ID       : POL000001
UPDATE records  : 1


In [74]:
# ---------------------------------------------------------
# STEP 18C - VERIFY SILVER AFTER POLICY UPDATE
# ---------------------------------------------------------

silver_after_update_df = spark.table(TARGET_TABLE)

silver_count_after_update = silver_after_update_df.count()

test_policy_after_merge_df = (
    silver_after_update_df
    .filter(F.col("policy_id") == TEST_POLICY_ID)
)

test_policy_after_merge_count = test_policy_after_merge_df.count()

updated_policy_row = test_policy_after_merge_df.first()

print("Silver post-MERGE verification.")
print("----------------------------------------")
print(f"Silver rows before : {silver_count_before_update}")
print(f"Silver rows after  : {silver_count_after_update}")
print(f"Net row increase   : {silver_count_after_update - silver_count_before_update}")
print(f"Test Policy rows   : {test_policy_after_merge_count}")
print(f"Updated premium    : {updated_policy_row['annual_premium']}")
print(f"Updated timestamp  : {updated_policy_row['last_updated']}")

assert silver_count_after_update == silver_count_before_update, \
    "Silver row count changed during Policy UPDATE."

assert test_policy_after_merge_count == 1, \
    f"Expected exactly one {TEST_POLICY_ID} in Silver."

assert float(updated_policy_row["annual_premium"]) == float(TEST_NEW_PREMIUM), \
    "Policy annual premium was not updated correctly."

display(
    test_policy_after_merge_df.select(
        "policy_id",
        "product_type",
        "annual_premium",
        "policy_status",
        "last_updated",
        "_silver_processed_ts"
    )
)

print()
print("Controlled Policy Silver UPDATE verification PASSED.")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 76, Finished, Available, Finished, False)

Silver post-MERGE verification.
----------------------------------------
Silver rows before : 750
Silver rows after  : 750
Net row increase   : 0
Test Policy rows   : 1
Updated premium    : 527.01
Updated timestamp  : 2027-12-09 00:00:00


SynapseWidget(Synapse.DataFrame, 23fb46bb-a6d6-4ca9-9d31-81716ca85bb0)


Controlled Policy Silver UPDATE verification PASSED.


## Step 19 — Audit Successful Policy UPDATE and Advance Watermark

The controlled Policy UPDATE has been successfully written and verified in Silver.

The pipeline will now complete the processing transaction by:

1. Writing a SUCCESS audit record for the incremental Policy execution.
2. Recording the source, insert, update, and reject counts.
3. Advancing the Policy watermark only after successful Silver processing.
4. Verifying the new stored watermark.
5. Confirming that restarting the incremental read returns zero records.

For this execution:

- Source records: **1**
- Inserts: **0**
- Updates: **1**
- Rejects: **0**
- Status: **SUCCESS**
- Previous watermark: **2027-12-08**
- New watermark: **2027-12-09**

The watermark is intentionally updated only after the Silver MERGE succeeds.
This prevents failed records from being skipped during a subsequent retry.

In [75]:

# ---------------------------------------------------------
# STEP 19A - WRITE POLICY UPDATE SUCCESS AUDIT
# ---------------------------------------------------------

UPDATE_BATCH_ID = str(uuid.uuid4())
UPDATE_AUDIT_START_TIME = datetime.now()

update_audit_end_time = datetime.now()

update_audit_record = [
    (
        UPDATE_BATCH_ID,
        PIPELINE_NAME,
        SOURCE_NAME,
        UPDATE_AUDIT_START_TIME,
        update_audit_end_time,
        incremental_silver_ready_count,  # source_count = 1
        test_insert_count,               # insert_count = 0
        test_update_count,               # update_count = 1
        incremental_reject_count,        # reject_count = 0
        "SUCCESS",
        None
    )
]

update_audit_df = spark.createDataFrame(
    update_audit_record,
    schema=audit_schema
)

(
    update_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

print("Policy UPDATE audit record written.")
print("----------------------------------------")
print(f"Batch ID      : {UPDATE_BATCH_ID}")
print(f"Source count  : {incremental_silver_ready_count}")
print(f"Insert count  : {test_insert_count}")
print(f"Update count  : {test_update_count}")
print(f"Reject count  : {incremental_reject_count}")
print("Status        : SUCCESS")

display(update_audit_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 77, Finished, Available, Finished, False)

Policy UPDATE audit record written.
----------------------------------------
Batch ID      : 5e5bfa17-7586-44a2-810b-b0073a3b2c78
Source count  : 1
Insert count  : 0
Update count  : 1
Reject count  : 0
Status        : SUCCESS


SynapseWidget(Synapse.DataFrame, 692fa32c-4372-434e-944a-6fcd30272322)

In [76]:

# ---------------------------------------------------------
# STEP 19B - ADVANCE POLICY WATERMARK AFTER SUCCESS
# ---------------------------------------------------------

NEW_POLICY_WATERMARK = (
    incremental_silver_ready_df
    .agg(
        F.max(F.col("last_updated"))
        .alias("new_watermark")
    )
    .first()["new_watermark"]
)

print("Preparing Policy watermark advancement.")
print("----------------------------------------")
print(f"Previous watermark : {LAST_WATERMARK}")
print(f"New watermark      : {NEW_POLICY_WATERMARK}")

assert NEW_POLICY_WATERMARK is not None, \
    "New Policy watermark cannot be NULL."

assert NEW_POLICY_WATERMARK > LAST_WATERMARK, \
    "New Policy watermark must be greater than previous watermark."


control_delta = DeltaTable.forName(
    spark,
    CONTROL_TABLE
)

(
    control_delta.alias("tgt")
    .merge(
        spark.createDataFrame(
            [
                (
                    SOURCE_NAME,
                    NEW_POLICY_WATERMARK
                )
            ],
            ["source_name", "new_watermark"]
        ).alias("src"),

        "tgt.source_name = src.source_name AND tgt.is_active = true"
    )
    .whenMatchedUpdate(
        set={
            "last_watermark": "src.new_watermark",
            "_updated_ts": "current_timestamp()"
        }
    )
    .execute()
)

print()
print("Policy watermark advancement completed.")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 78, Finished, Available, Finished, False)

Preparing Policy watermark advancement.
----------------------------------------
Previous watermark : 1900-01-01 00:00:00
New watermark      : 2027-12-09 00:00:00

Policy watermark advancement completed.


In [77]:
# ---------------------------------------------------------
# STEP 19C - VERIFY UPDATED POLICY WATERMARK
# ---------------------------------------------------------

updated_policy_control_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .select(
        "source_name",
        "watermark_column",
        "last_watermark",
        "load_type",
        "is_active",
        "_updated_ts"
    )
)

display(updated_policy_control_df)

updated_policy_control = updated_policy_control_df.first()

assert updated_policy_control["last_watermark"] == NEW_POLICY_WATERMARK, \
    "Policy watermark was not updated correctly."

print()
print("Policy watermark verification PASSED.")
print(
    f"Stored watermark : "
    f"{updated_policy_control['last_watermark']}"
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 79, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d96629f2-ad1e-40d2-868f-342b95506eaa)


Policy watermark verification PASSED.
Stored watermark : 2027-12-09 00:00:00



## Step 20 — Verify Restart and Idempotent Processing

The successful Policy UPDATE has been merged into Silver and the Policy
watermark has advanced to `2027-12-09`.

This final test simulates the next pipeline execution without adding any
new Bronze Policy records.

The pipeline should:

1. Read the stored Policy watermark.
2. Scan Bronze using `last_updated > stored watermark`.
3. Return zero incremental records.
4. Confirm that the previously processed Policy is not processed again.

### Expected Result

- Stored watermark: **2027-12-09**
- Incremental records: **0**
- `POL000001` is not selected again.

This proves that the Policy incremental pipeline is restart-safe and
idempotent after a successful UPDATE.

In [78]:
# ---------------------------------------------------------
# STEP 20 - POLICY UPDATE RESTART / IDEMPOTENCY TEST
# ---------------------------------------------------------

restart_policy_config = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .first()
)

RESTART_POLICY_WATERMARK = restart_policy_config["last_watermark"]
RESTART_WATERMARK_COLUMN = restart_policy_config["watermark_column"]

restart_bronze_policy_df = spark.table(SOURCE_TABLE)

restart_incremental_policy_df = (
    restart_bronze_policy_df
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(RESTART_WATERMARK_COLUMN))
    )
    .filter(
        F.col("_watermark_ts") > F.lit(RESTART_POLICY_WATERMARK)
    )
)

restart_incremental_count = restart_incremental_policy_df.count()

print("Policy UPDATE restart test completed.")
print("----------------------------------------")
print(f"Bronze Policy rows  : {restart_bronze_policy_df.count()}")
print(f"Stored watermark    : {RESTART_POLICY_WATERMARK}")
print(f"Incremental records : {restart_incremental_count}")

assert restart_incremental_count == 0, \
    (
        "Restart/idempotency failure. "
        f"Expected 0 incremental Policies, "
        f"found {restart_incremental_count}."
    )

print()
print("Policy UPDATE restart/idempotency PASSED.")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 80, Finished, Available, Finished, False)

Policy UPDATE restart test completed.
----------------------------------------
Bronze Policy rows  : 752
Stored watermark    : 2027-12-09 00:00:00
Incremental records : 0

Policy UPDATE restart/idempotency PASSED.



# Step 21 — Production Policy Incremental Processing

The Policy incremental-processing logic has now been validated through
controlled INSERT, UPDATE, NO-OP, failure, watermark, audit, and restart tests.

This section consolidates those validated components into a production-style
execution flow.

## Processing Architecture

The Policy pipeline follows this sequence:

`Configuration`
→ `Read Control Metadata`
→ `Incremental Bronze Read`
→ `Validate`
→ `Reject Invalid Records`
→ `Deduplicate`
→ `Standardize / Transform`
→ `Compare with Silver`
→ `INSERT / UPDATE / NO-OP Detection`
→ `Delta MERGE`
→ `Audit`
→ `Advance Watermark`
→ `Restart-Safe Completion`

## Transaction Principle

The watermark is advanced only after successful Silver processing.

If processing fails:

- The execution is written to the audit table with `FAILED` status.
- The error message is captured.
- The watermark is not advanced.
- The next execution can safely retry the same incremental data.

If no new source data exists:

- No Silver MERGE is executed.
- A `NO_DATA` audit record is written.
- The watermark remains unchanged.

This design provides incremental processing, idempotency, observability,
failure recovery, and controlled data-quality handling.


In [79]:
# ---------------------------------------------------------
# STEP 21A - PRODUCTION PIPELINE CONFIGURATION
# ---------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

from datetime import datetime
import uuid


# ---------------------------------------------------------
# PIPELINE IDENTITY
# ---------------------------------------------------------

PROD_PIPELINE_NAME = "PL_Insurance_Medallion_ETL"
PROD_SOURCE_NAME = "POLICIES"


# ---------------------------------------------------------
# TABLES
# Reuse the validated table variables from Steps 1-20
# ---------------------------------------------------------

PROD_SOURCE_TABLE = SOURCE_TABLE
PROD_TARGET_TABLE = TARGET_TABLE
PROD_CONTROL_TABLE = CONTROL_TABLE
PROD_AUDIT_TABLE = AUDIT_TABLE


# ---------------------------------------------------------
# EXECUTION IDENTITY
# ---------------------------------------------------------

PROD_BATCH_ID = str(uuid.uuid4())
PROD_RUN_START_TIME = datetime.now()


print("Production Policy pipeline initialized.")
print("----------------------------------------")
print(f"Pipeline      : {PROD_PIPELINE_NAME}")
print(f"Source        : {PROD_SOURCE_NAME}")
print(f"Batch ID      : {PROD_BATCH_ID}")
print(f"Run start     : {PROD_RUN_START_TIME}")
print(f"Bronze table  : {PROD_SOURCE_TABLE}")
print(f"Silver table  : {PROD_TARGET_TABLE}")
print(f"Control table : {PROD_CONTROL_TABLE}")
print(f"Audit table   : {PROD_AUDIT_TABLE}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 81, Finished, Available, Finished, False)

Production Policy pipeline initialized.
----------------------------------------
Pipeline      : PL_Insurance_Medallion_ETL
Source        : POLICIES
Batch ID      : 2781b4a9-c5d4-4165-adbd-c88aa571774d
Run start     : 2026-08-21 20:00:56.129481
Bronze table  : LH_Bronze.dbo.bronze_policies
Silver table  : LH_Silver.dbo.silver_policies
Control table : LH_Silver.dbo.etl_control
Audit table   : LH_Silver.dbo.etl_batch_audit


## Step 21B — Read and Validate Policy Control Configuration

The production pipeline retrieves the active Policy configuration from the
ETL control table.

Before processing any Bronze data, the pipeline verifies that:

1. Exactly one active configuration exists for `POLICIES`.
2. The configured source table is available.
3. The configured target table is available.
4. A watermark column is defined.
5. The current stored watermark is available.
6. The load type is configured as expected.

This metadata-driven approach separates pipeline processing logic from
source-specific configuration and allows the same processing pattern to be
extended to additional insurance entities.

In [80]:
# ---------------------------------------------------------
# STEP 21B - READ AND VALIDATE PRODUCTION CONTROL CONFIG
# ---------------------------------------------------------

prod_policy_config_df = (
    spark.table(PROD_CONTROL_TABLE)
    .filter(
        (F.col("source_name") == PROD_SOURCE_NAME)
        & (F.col("is_active") == True)
    )
)

prod_config_count = prod_policy_config_df.count()

assert prod_config_count == 1, (
    f"Expected exactly one active configuration for "
    f"{PROD_SOURCE_NAME}, found {prod_config_count}."
)

prod_policy_config = prod_policy_config_df.first()


# ---------------------------------------------------------
# EXTRACT CONFIGURATION
# ---------------------------------------------------------

PROD_CONFIG_SOURCE_TABLE = prod_policy_config["source_table"]
PROD_CONFIG_TARGET_TABLE = prod_policy_config["target_table"]
PROD_WATERMARK_COLUMN = prod_policy_config["watermark_column"]
PROD_LAST_WATERMARK = prod_policy_config["last_watermark"]
PROD_LOAD_TYPE = prod_policy_config["load_type"]


# ---------------------------------------------------------
# DEFENSIVE CONFIGURATION CHECKS
# ---------------------------------------------------------

assert PROD_CONFIG_SOURCE_TABLE is not None, \
    "Configured source table cannot be NULL."

assert PROD_CONFIG_TARGET_TABLE is not None, \
    "Configured target table cannot be NULL."

assert PROD_WATERMARK_COLUMN is not None, \
    "Configured watermark column cannot be NULL."

assert PROD_LAST_WATERMARK is not None, \
    "Stored watermark cannot be NULL."

assert PROD_LOAD_TYPE is not None, \
    "Load type cannot be NULL."


print("Production Policy configuration loaded.")
print("----------------------------------------")
print(f"Source name       : {PROD_SOURCE_NAME}")
print(f"Source table      : {PROD_CONFIG_SOURCE_TABLE}")
print(f"Target table      : {PROD_CONFIG_TARGET_TABLE}")
print(f"Watermark column  : {PROD_WATERMARK_COLUMN}")
print(f"Stored watermark  : {PROD_LAST_WATERMARK}")
print(f"Load type         : {PROD_LOAD_TYPE}")

display(prod_policy_config_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 82, Finished, Available, Finished, False)

Production Policy configuration loaded.
----------------------------------------
Source name       : POLICIES
Source table      : LH_Bronze.dbo.bronze_policies
Target table      : LH_Silver.dbo.silver_policies
Watermark column  : last_updated
Stored watermark  : 2027-12-09 00:00:00
Load type         : INCREMENTAL


SynapseWidget(Synapse.DataFrame, 0c258373-8a61-4e5b-bf0d-ef4e50fd4450)

## Step 21C — Read Incremental Policy Data from Bronze

The production pipeline now reads the Bronze Policy table using the
watermark stored in the ETL control table.

Only records where:

`last_updated > stored watermark`

are eligible for processing.

The pipeline also creates a normalized timestamp column named
`_watermark_ts` for consistent incremental filtering.

### Expected Result for the Current Run

The stored Policy watermark is `2027-12-09`.

Because no newer Policy records have been added since the controlled
UPDATE test, this execution should return:

- Bronze rows: **752**
- Incremental rows: **0**

This represents the normal `NO_DATA` execution path of an incremental
production pipeline.

In [81]:
# ---------------------------------------------------------
# STEP 21C - READ PRODUCTION INCREMENTAL BRONZE POLICIES
# ---------------------------------------------------------

prod_bronze_policy_df = spark.table(PROD_CONFIG_SOURCE_TABLE)

prod_bronze_count = prod_bronze_policy_df.count()


# Validate configured watermark column exists
assert PROD_WATERMARK_COLUMN in prod_bronze_policy_df.columns, (
    f"Configured watermark column '{PROD_WATERMARK_COLUMN}' "
    f"does not exist in {PROD_CONFIG_SOURCE_TABLE}."
)


# Normalize watermark to timestamp
prod_bronze_with_watermark_df = (
    prod_bronze_policy_df
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(PROD_WATERMARK_COLUMN))
    )
)


# Incremental read
prod_incremental_policy_df = (
    prod_bronze_with_watermark_df
    .filter(
        F.col("_watermark_ts") > F.lit(PROD_LAST_WATERMARK)
    )
)

prod_incremental_count = prod_incremental_policy_df.count()


print("Production Policy incremental read completed.")
print("----------------------------------------")
print(f"Bronze rows         : {prod_bronze_count}")
print(f"Stored watermark    : {PROD_LAST_WATERMARK}")
print(f"Watermark column    : {PROD_WATERMARK_COLUMN}")
print(f"Incremental records : {prod_incremental_count}")

display(
    prod_incremental_policy_df
    .orderBy(F.col("_watermark_ts").desc())
    .limit(20)
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 83, Finished, Available, Finished, False)

Production Policy incremental read completed.
----------------------------------------
Bronze rows         : 752
Stored watermark    : 2027-12-09 00:00:00
Watermark column    : last_updated
Incremental records : 0


SynapseWidget(Synapse.DataFrame, a4f6824c-7a15-4917-963e-9f17f8712070)


## Step 21D — Handle the NO_DATA Execution Path

An incremental pipeline must treat the absence of new source records as a
successful operational condition rather than a processing failure.

When no Policy records exist beyond the stored watermark, the pipeline:

1. Skips validation, transformation, and Delta MERGE processing.
2. Writes one audit record with status `NO_DATA`.
3. Records zero source, insert, update, and reject counts.
4. Leaves the Policy watermark unchanged.
5. Completes the execution successfully.

This prevents unnecessary Silver processing while maintaining a complete
operational audit trail for every pipeline execution.

In [82]:
# ---------------------------------------------------------
# STEP 21D - PRODUCTION NO_DATA HANDLING
# ---------------------------------------------------------

PROD_HAS_DATA = prod_incremental_count > 0

if not PROD_HAS_DATA:

    prod_no_data_end_time = datetime.now()

    prod_no_data_record = [
        (
            PROD_BATCH_ID,
            PROD_PIPELINE_NAME,
            PROD_SOURCE_NAME,
            PROD_RUN_START_TIME,
            prod_no_data_end_time,
            0,          # source_count
            0,          # insert_count
            0,          # update_count
            0,          # reject_count
            "NO_DATA",
            None
        )
    ]

    prod_no_data_audit_df = spark.createDataFrame(
        prod_no_data_record,
        schema=audit_schema
    )

    (
        prod_no_data_audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(PROD_AUDIT_TABLE)
    )

    print("Production Policy NO_DATA path completed.")
    print("----------------------------------------")
    print(f"Batch ID         : {PROD_BATCH_ID}")
    print(f"Source           : {PROD_SOURCE_NAME}")
    print(f"Incremental rows : {prod_incremental_count}")
    print("Status           : NO_DATA")
    print(f"Watermark        : {PROD_LAST_WATERMARK}")
    print()
    print("Silver MERGE skipped.")
    print("Watermark advancement skipped.")

else:

    print("Production Policy data detected.")
    print("----------------------------------------")
    print(f"Incremental rows : {prod_incremental_count}")
    print("Continue to validation and processing.")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 84, Finished, Available, Finished, False)

Production Policy NO_DATA path completed.
----------------------------------------
Batch ID         : 2781b4a9-c5d4-4165-adbd-c88aa571774d
Source           : POLICIES
Incremental rows : 0
Status           : NO_DATA
Watermark        : 2027-12-09 00:00:00

Silver MERGE skipped.
Watermark advancement skipped.


## Step 21E — Verify NO_DATA Audit and Watermark Protection

This step verifies the operational guarantees of the production `NO_DATA`
execution path.

The pipeline must confirm that:

1. Exactly one audit record exists for the current batch.
2. The audit status is `NO_DATA`.
3. Source, insert, update, and reject counts are zero.
4. The Policy watermark remains unchanged.
5. No Silver processing was required.

This confirms that an empty incremental execution is handled successfully
without modifying the processing state.

In [83]:
# ---------------------------------------------------------
# STEP 21E - VERIFY PRODUCTION NO_DATA EXECUTION
# ---------------------------------------------------------

prod_no_data_audit_check_df = (
    spark.table(PROD_AUDIT_TABLE)
    .filter(F.col("batch_id") == PROD_BATCH_ID)
)

prod_no_data_audit_count = prod_no_data_audit_check_df.count()

assert prod_no_data_audit_count == 1, (
    f"Expected exactly one audit record for batch "
    f"{PROD_BATCH_ID}, found {prod_no_data_audit_count}."
)

prod_no_data_audit = prod_no_data_audit_check_df.first()


# ---------------------------------------------------------
# VERIFY AUDIT VALUES
# ---------------------------------------------------------

assert prod_no_data_audit["status"] == "NO_DATA", \
    "Expected audit status NO_DATA."

assert prod_no_data_audit["source_count"] == 0, \
    "Expected source_count = 0."

assert prod_no_data_audit["insert_count"] == 0, \
    "Expected insert_count = 0."

assert prod_no_data_audit["update_count"] == 0, \
    "Expected update_count = 0."

assert prod_no_data_audit["reject_count"] == 0, \
    "Expected reject_count = 0."


# ---------------------------------------------------------
# VERIFY WATERMARK DID NOT CHANGE
# ---------------------------------------------------------

prod_control_after_no_data = (
    spark.table(PROD_CONTROL_TABLE)
    .filter(
        (F.col("source_name") == PROD_SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .first()
)

prod_watermark_after_no_data = (
    prod_control_after_no_data["last_watermark"]
)

assert prod_watermark_after_no_data == PROD_LAST_WATERMARK, (
    "Policy watermark changed during NO_DATA execution."
)


# ---------------------------------------------------------
# DISPLAY RESULTS
# ---------------------------------------------------------

display(prod_no_data_audit_check_df)

print()
print("Production Policy NO_DATA verification PASSED.")
print("----------------------------------------")
print(f"Batch ID          : {PROD_BATCH_ID}")
print(f"Audit status      : {prod_no_data_audit['status']}")
print(f"Source count      : {prod_no_data_audit['source_count']}")
print(f"Insert count      : {prod_no_data_audit['insert_count']}")
print(f"Update count      : {prod_no_data_audit['update_count']}")
print(f"Reject count      : {prod_no_data_audit['reject_count']}")
print(f"Original watermark: {PROD_LAST_WATERMARK}")
print(f"Current watermark : {prod_watermark_after_no_data}")

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 85, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6c04da3a-d87e-4864-bd04-72eff4569cbf)


Production Policy NO_DATA verification PASSED.
----------------------------------------
Batch ID          : 2781b4a9-c5d4-4165-adbd-c88aa571774d
Audit status      : NO_DATA
Source count      : 0
Insert count      : 0
Update count      : 0
Reject count      : 0
Original watermark: 2027-12-09 00:00:00
Current watermark : 2027-12-09 00:00:00



# Step 22 — Production DATA Path Validation

The production `NO_DATA` execution path has been successfully validated.

The next test validates the production processing path when new incremental
Policy data becomes available.

A controlled new Policy will be appended to Bronze with a `last_updated`
timestamp greater than the current stored watermark.

The production pipeline must then:

1. Detect the new Policy through the watermark.
2. Validate the incoming record.
3. Deduplicate the incremental data.
4. Standardize and transform the Policy for Silver.
5. Identify the Policy as an INSERT.
6. MERGE the new Policy into the Silver Delta table.
7. Verify reconciliation and Silver persistence.
8. Write a `SUCCESS` audit record.
9. Advance the Policy watermark only after successful processing.
10. Confirm restart/idempotency by returning zero records on the next run.

This validates the complete successful production DATA execution path.

In [87]:
# ---------------------------------------------------------
# STEP 22A - CREATE CONTROLLED NEW POLICY
# PRESERVE BRONZE SOURCE DATA TYPES
# ---------------------------------------------------------

PROD_TEST_POLICY_ID = "POL999901"
PROD_TEST_LAST_UPDATED = "2027-12-10"

bronze_policy_table_df = spark.table(PROD_CONFIG_SOURCE_TABLE)

# Verify test Policy does not already exist
existing_prod_test_count = (
    bronze_policy_table_df
    .filter(F.col("policy_id") == PROD_TEST_POLICY_ID)
    .count()
)

assert existing_prod_test_count == 0, (
    f"{PROD_TEST_POLICY_ID} already exists in Bronze."
)

# Read the actual Bronze schema
bronze_schema_map = {
    field.name: field.dataType.simpleString()
    for field in bronze_policy_table_df.schema.fields
}

# Start with one existing Bronze row so all untouched
# columns already have the correct source representation
prod_test_policy_df = (
    bronze_policy_table_df
    .limit(1)

    .withColumn(
        "policy_id",
        F.lit(PROD_TEST_POLICY_ID)
        .cast(bronze_schema_map["policy_id"])
    )

    .withColumn(
        "customer_id",
        F.lit("CUST00999")
        .cast(bronze_schema_map["customer_id"])
    )

    .withColumn(
        "product_type",
        F.lit("Auto")
        .cast(bronze_schema_map["product_type"])
    )

    .withColumn(
        "annual_premium",
        F.lit("1250.00")
        .cast(bronze_schema_map["annual_premium"])
    )

    .withColumn(
        "coverage_limit",
        F.lit("250000")
        .cast(bronze_schema_map["coverage_limit"])
    )

    .withColumn(
        "deductible",
        F.lit("1000")
        .cast(bronze_schema_map["deductible"])
    )

    .withColumn(
        "policy_status",
        F.lit("Active")
        .cast(bronze_schema_map["policy_status"])
    )

    .withColumn(
        "agent_id",
        F.lit("AGT999")
        .cast(bronze_schema_map["agent_id"])
    )

    .withColumn(
        "last_updated",
        F.lit(PROD_TEST_LAST_UPDATED)
        .cast(bronze_schema_map["last_updated"])
    )
)

print("Controlled production INSERT Policy created.")
print("----------------------------------------")
print(f"Policy ID       : {PROD_TEST_POLICY_ID}")
print(f"Last updated    : {PROD_TEST_LAST_UPDATED}")
print(f"Stored watermark: {PROD_LAST_WATERMARK}")

prod_test_policy_df.printSchema()

display(prod_test_policy_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 89, Finished, Available, Finished, False)

Controlled production INSERT Policy created.
----------------------------------------
Policy ID       : POL999901
Last updated    : 2027-12-10
Stored watermark: 2027-12-09 00:00:00
root
 |-- policy_id: string (nullable = false)
 |-- customer_id: string (nullable = false)
 |-- product_type: string (nullable = false)
 |-- policy_start_date: string (nullable = true)
 |-- policy_end_date: string (nullable = true)
 |-- annual_premium: string (nullable = false)
 |-- coverage_limit: string (nullable = false)
 |-- deductible: string (nullable = false)
 |-- policy_status: string (nullable = false)
 |-- agent_id: string (nullable = false)
 |-- last_updated: string (nullable = false)



SynapseWidget(Synapse.DataFrame, b5d8ceb4-99aa-4f5b-bacf-75fe0583f4ef)


## Step 22B — Append Controlled Policy to Bronze

The controlled Policy record has been validated and is now appended to the
Bronze Policy Delta table.

This simulates the arrival of a new Policy from an upstream source system.

After the append:

- Bronze row count should increase from **752 to 753**.
- `POL999901` should occur exactly once.
- Its `last_updated` value should be **2027-12-10**.
- The stored control watermark remains **2027-12-09**.

The record therefore becomes eligible for the next production incremental
execution.

In [88]:

# ---------------------------------------------------------
# STEP 22B - APPEND CONTROLLED NEW POLICY TO BRONZE
# ---------------------------------------------------------

prod_bronze_count_before_insert = (
    spark.table(PROD_CONFIG_SOURCE_TABLE).count()
)

# Align exactly to Bronze schema before append
bronze_columns = spark.table(PROD_CONFIG_SOURCE_TABLE).columns

prod_test_policy_write_df = (
    prod_test_policy_df
    .select(*bronze_columns)
)

(
    prod_test_policy_write_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(PROD_CONFIG_SOURCE_TABLE)
)

prod_bronze_count_after_insert = (
    spark.table(PROD_CONFIG_SOURCE_TABLE).count()
)

prod_inserted_policy_df = (
    spark.table(PROD_CONFIG_SOURCE_TABLE)
    .filter(F.col("policy_id") == PROD_TEST_POLICY_ID)
)

prod_inserted_policy_count = prod_inserted_policy_df.count()


print("Controlled production Policy appended to Bronze.")
print("----------------------------------------")
print(f"Policy ID          : {PROD_TEST_POLICY_ID}")
print(f"Bronze rows before : {prod_bronze_count_before_insert}")
print(f"Bronze rows after  : {prod_bronze_count_after_insert}")
print(
    f"Net increase       : "
    f"{prod_bronze_count_after_insert - prod_bronze_count_before_insert}"
)
print(f"Policy occurrences : {prod_inserted_policy_count}")


assert (
    prod_bronze_count_after_insert
    == prod_bronze_count_before_insert + 1
), "Expected Bronze Policy table to increase by exactly one row."

assert prod_inserted_policy_count == 1, (
    f"Expected exactly one {PROD_TEST_POLICY_ID} record in Bronze, "
    f"found {prod_inserted_policy_count}."
)

display(prod_inserted_policy_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 90, Finished, Available, Finished, False)

Controlled production Policy appended to Bronze.
----------------------------------------
Policy ID          : POL999901
Bronze rows before : 752
Bronze rows after  : 753
Net increase       : 1
Policy occurrences : 1


SynapseWidget(Synapse.DataFrame, 394c5928-2643-4dc1-8dd0-66fb1405ee1a)


## Step 22C — Start New Production Batch and Read Incremental Data

A new Policy has arrived in Bronze after the previous production execution.

This step starts a new production batch and reloads the current control
watermark before reading Bronze.

The pipeline must detect only records where:

`last_updated > stored watermark`

### Expected Result

- Bronze rows: **753**
- Stored watermark: **2027-12-09**
- Incremental records: **1**
- Incremental Policy: **POL999901**
- Incoming watermark: **2027-12-10**

This begins the production DATA processing path.

In [90]:
# ---------------------------------------------------------
# STEP 22C - START NEW PRODUCTION DATA BATCH
# ---------------------------------------------------------

PROD_BATCH_ID = str(uuid.uuid4())
PROD_RUN_START_TIME = datetime.now()

print("New production DATA batch initialized.")
print("----------------------------------------")
print(f"Pipeline  : {PROD_PIPELINE_NAME}")
print(f"Source    : {PROD_SOURCE_NAME}")
print(f"Batch ID  : {PROD_BATCH_ID}")
print(f"Run start : {PROD_RUN_START_TIME}")


# ---------------------------------------------------------
# RELOAD CURRENT CONTROL STATE
# ---------------------------------------------------------

prod_data_config = (
    spark.table(PROD_CONTROL_TABLE)
    .filter(
        (F.col("source_name") == PROD_SOURCE_NAME)
        & (F.col("is_active") == True)
    )
    .first()
)

PROD_DATA_WATERMARK_COLUMN = prod_data_config["watermark_column"]
PROD_DATA_LAST_WATERMARK = prod_data_config["last_watermark"]

print()
print("Current control state loaded.")
print("----------------------------------------")
print(f"Watermark column : {PROD_DATA_WATERMARK_COLUMN}")
print(f"Stored watermark : {PROD_DATA_LAST_WATERMARK}")


# ---------------------------------------------------------
# READ BRONZE INCREMENTALLY
# ---------------------------------------------------------

prod_data_bronze_df = spark.table(PROD_CONFIG_SOURCE_TABLE)

prod_data_bronze_count = prod_data_bronze_df.count()

prod_data_incremental_df = (
    prod_data_bronze_df

    .withColumn(
        "_watermark_ts",
        F.to_timestamp(
            F.col(PROD_DATA_WATERMARK_COLUMN)
        )
    )

    .filter(
        F.col("_watermark_ts")
        > F.lit(PROD_DATA_LAST_WATERMARK)
    )
)

prod_data_incremental_count = prod_data_incremental_df.count()


# ---------------------------------------------------------
# VERIFY CONTROLLED DATA TEST
# ---------------------------------------------------------

assert prod_data_incremental_count == 1, (
    "Controlled production DATA test expected exactly "
    f"1 incremental Policy, found {prod_data_incremental_count}."
)

prod_detected_test_count = (
    prod_data_incremental_df
    .filter(F.col("policy_id") == PROD_TEST_POLICY_ID)
    .count()
)

assert prod_detected_test_count == 1, (
    f"Expected incremental Policy {PROD_TEST_POLICY_ID} "
    "was not detected."
)


print()
print("Production incremental DATA detected.")
print("----------------------------------------")
print(f"Bronze rows         : {prod_data_bronze_count}")
print(f"Stored watermark    : {PROD_DATA_LAST_WATERMARK}")
print(f"Incremental records : {prod_data_incremental_count}")
print(f"Test Policy         : {PROD_TEST_POLICY_ID}")

display(
    prod_data_incremental_df
    .orderBy(F.col("_watermark_ts").desc())
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 92, Finished, Available, Finished, False)

New production DATA batch initialized.
----------------------------------------
Pipeline  : PL_Insurance_Medallion_ETL
Source    : POLICIES
Batch ID  : 3be3bb42-04af-4ea5-ae98-7ec5812db192
Run start : 2026-08-21 20:11:37.429348

Current control state loaded.
----------------------------------------
Watermark column : last_updated
Stored watermark : 2027-12-09 00:00:00

Production incremental DATA detected.
----------------------------------------
Bronze rows         : 753
Stored watermark    : 2027-12-09 00:00:00
Incremental records : 1
Test Policy         : POL999901


SynapseWidget(Synapse.DataFrame, 5b9176c5-0e0c-43b1-93aa-fafcade7d60f)


## Step 22D — Validate Incremental Policy Records

The production pipeline validates incremental Policy records before they are
allowed to enter the Silver layer.

Validation protects Silver from malformed or incomplete source data.

The following business and data-quality rules are applied:

1. `policy_id` must be present.
2. `customer_id` must be present.
3. `product_type` must be present.
4. `policy_start_date` must be a valid date.
5. `policy_end_date` must be a valid date.
6. Policy end date must not be earlier than policy start date.
7. `annual_premium` must be numeric and non-negative.
8. `coverage_limit` must be numeric and non-negative.
9. `deductible` must be numeric and non-negative.
10. `policy_status` must be present.
11. `last_updated` must contain a valid timestamp.

Records that satisfy all validation rules continue through Silver processing.

Invalid records are separated into the reject path with a descriptive
`reject_reason`.

For the controlled production test, `POL999901` should pass validation.


In [91]:
# ---------------------------------------------------------
# STEP 22D - PRODUCTION POLICY VALIDATION
# ---------------------------------------------------------

prod_policy_validation_df = (
    prod_data_incremental_df

    # Parse / cast source values
    .withColumn(
        "_parsed_start_date",
        F.to_date(F.col("policy_start_date"))
    )

    .withColumn(
        "_parsed_end_date",
        F.to_date(F.col("policy_end_date"))
    )

    .withColumn(
        "_parsed_last_updated",
        F.to_timestamp(F.col("last_updated"))
    )

    .withColumn(
        "_parsed_annual_premium",
        F.col("annual_premium").cast("double")
    )

    .withColumn(
        "_parsed_coverage_limit",
        F.col("coverage_limit").cast("double")
    )

    .withColumn(
        "_parsed_deductible",
        F.col("deductible").cast("double")
    )
)


# ---------------------------------------------------------
# BUILD REJECT REASON
# ---------------------------------------------------------

prod_policy_validation_df = (
    prod_policy_validation_df
    .withColumn(
        "reject_reason",

        F.when(
            F.col("policy_id").isNull()
            | (F.trim(F.col("policy_id")) == ""),
            F.lit("MISSING_POLICY_ID")
        )

        .when(
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id")) == ""),
            F.lit("MISSING_CUSTOMER_ID")
        )

        .when(
            F.col("product_type").isNull()
            | (F.trim(F.col("product_type")) == ""),
            F.lit("MISSING_PRODUCT_TYPE")
        )

        .when(
            F.col("_parsed_start_date").isNull(),
            F.lit("INVALID_POLICY_START_DATE")
        )

        .when(
            F.col("_parsed_end_date").isNull(),
            F.lit("INVALID_POLICY_END_DATE")
        )

        .when(
            F.col("_parsed_end_date")
            < F.col("_parsed_start_date"),
            F.lit("END_DATE_BEFORE_START_DATE")
        )

        .when(
            F.col("_parsed_annual_premium").isNull(),
            F.lit("INVALID_ANNUAL_PREMIUM")
        )

        .when(
            F.col("_parsed_annual_premium") < 0,
            F.lit("NEGATIVE_ANNUAL_PREMIUM")
        )

        .when(
            F.col("_parsed_coverage_limit").isNull(),
            F.lit("INVALID_COVERAGE_LIMIT")
        )

        .when(
            F.col("_parsed_coverage_limit") < 0,
            F.lit("NEGATIVE_COVERAGE_LIMIT")
        )

        .when(
            F.col("_parsed_deductible").isNull(),
            F.lit("INVALID_DEDUCTIBLE")
        )

        .when(
            F.col("_parsed_deductible") < 0,
            F.lit("NEGATIVE_DEDUCTIBLE")
        )

        .when(
            F.col("policy_status").isNull()
            | (F.trim(F.col("policy_status")) == ""),
            F.lit("MISSING_POLICY_STATUS")
        )

        .when(
            F.col("_parsed_last_updated").isNull(),
            F.lit("INVALID_LAST_UPDATED")
        )

        .otherwise(F.lit(None))
    )
)


# ---------------------------------------------------------
# SPLIT VALID / REJECTED RECORDS
# ---------------------------------------------------------

prod_valid_policy_df = (
    prod_policy_validation_df
    .filter(F.col("reject_reason").isNull())
)

prod_rejected_policy_df = (
    prod_policy_validation_df
    .filter(F.col("reject_reason").isNotNull())
)

prod_valid_count = prod_valid_policy_df.count()
prod_reject_count = prod_rejected_policy_df.count()


# ---------------------------------------------------------
# RECONCILIATION
# ---------------------------------------------------------

assert (
    prod_valid_count + prod_reject_count
    == prod_data_incremental_count
), "Production Policy validation reconciliation failed."


print("Production Policy validation completed.")
print("----------------------------------------")
print(f"Incremental records : {prod_data_incremental_count}")
print(f"Valid records       : {prod_valid_count}")
print(f"Rejected records    : {prod_reject_count}")
print(
    f"Reconciliation      : "
    f"{prod_valid_count + prod_reject_count}"
)

display(
    prod_valid_policy_df.select(
        "policy_id",
        "customer_id",
        "product_type",
        "annual_premium",
        "coverage_limit",
        "deductible",
        "policy_status",
        "last_updated"
    )
)

if prod_reject_count > 0:
    print()
    print("Rejected Policy records:")
    
    display(
        prod_rejected_policy_df.select(
            "policy_id",
            "customer_id",
            "product_type",
            "annual_premium",
            "policy_status",
            "last_updated",
            "reject_reason"
        )
    )

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 93, Finished, Available, Finished, False)

Production Policy validation completed.
----------------------------------------
Incremental records : 1
Valid records       : 1
Rejected records    : 0
Reconciliation      : 1


SynapseWidget(Synapse.DataFrame, 4393349f-6bb5-4138-8fa5-454207a21347)


## Step 23 — Prepare Silver-Ready Policy and Classify Change Type

The validated incremental Policy records are now prepared for Silver processing.

This step performs three operations:

1. Deduplicate incremental Policy records by `policy_id`, keeping the newest
   `last_updated` version.
2. Transform the surviving records into the canonical Silver representation.
3. Compare the incoming records with Silver to classify each record as:

   - INSERT
   - UPDATE
   - NO-OP

For the controlled production test, `POL999901` is a new Policy and is expected
to be classified as:

**1 INSERT, 0 UPDATE, 0 NO-OP**

In [92]:

# ---------------------------------------------------------
# STEP 23 - DEDUPLICATE, TRANSFORM, CLASSIFY
# ---------------------------------------------------------

from pyspark.sql.window import Window

# Deduplicate by newest last_updated
prod_policy_window = (
    Window
    .partitionBy("policy_id")
    .orderBy(F.col("_parsed_last_updated").desc())
)

prod_ranked_policy_df = (
    prod_valid_policy_df
    .withColumn(
        "_policy_rank",
        F.row_number().over(prod_policy_window)
    )
)

prod_dedup_policy_df = (
    prod_ranked_policy_df
    .filter(F.col("_policy_rank") == 1)
)

prod_duplicate_policy_df = (
    prod_ranked_policy_df
    .filter(F.col("_policy_rank") > 1)
)

prod_duplicate_count = prod_duplicate_policy_df.count()


# Canonical Silver transformation
prod_silver_ready_policy_df = (
    prod_dedup_policy_df
    .select(
        F.col("policy_id"),
        F.col("customer_id"),

        F.upper(F.trim(F.col("product_type")))
            .alias("product_type"),

        F.col("_parsed_start_date")
            .alias("policy_start_date"),

        F.col("_parsed_end_date")
            .alias("policy_end_date"),

        F.col("_parsed_annual_premium")
            .alias("annual_premium"),

        F.col("_parsed_coverage_limit")
            .alias("coverage_limit"),

        F.col("_parsed_deductible")
            .alias("deductible"),

        F.upper(F.trim(F.col("policy_status")))
            .alias("policy_status"),

        F.col("agent_id"),

        F.col("_parsed_last_updated")
            .alias("last_updated"),

        F.current_timestamp()
            .alias("_silver_processed_ts")
    )
)

prod_silver_ready_count = prod_silver_ready_policy_df.count()


# Compare with existing Silver
prod_existing_silver_df = spark.table(PROD_CONFIG_TARGET_TABLE)

prod_comparison_df = (
    prod_silver_ready_policy_df.alias("src")
    .join(
        prod_existing_silver_df.alias("tgt"),
        F.col("src.policy_id") == F.col("tgt.policy_id"),
        "left"
    )
)

prod_change_condition = (
    ~F.col("src.customer_id").eqNullSafe(F.col("tgt.customer_id"))
    | ~F.col("src.product_type").eqNullSafe(F.col("tgt.product_type"))
    | ~F.col("src.policy_start_date").eqNullSafe(F.col("tgt.policy_start_date"))
    | ~F.col("src.policy_end_date").eqNullSafe(F.col("tgt.policy_end_date"))
    | ~F.col("src.annual_premium").eqNullSafe(F.col("tgt.annual_premium"))
    | ~F.col("src.coverage_limit").eqNullSafe(F.col("tgt.coverage_limit"))
    | ~F.col("src.deductible").eqNullSafe(F.col("tgt.deductible"))
    | ~F.col("src.policy_status").eqNullSafe(F.col("tgt.policy_status"))
    | ~F.col("src.agent_id").eqNullSafe(F.col("tgt.agent_id"))
)

prod_insert_df = (
    prod_comparison_df
    .filter(F.col("tgt.policy_id").isNull())
    .select("src.*")
)

prod_update_df = (
    prod_comparison_df
    .filter(
        F.col("tgt.policy_id").isNotNull()
        & prod_change_condition
    )
    .select("src.*")
)

prod_noop_df = (
    prod_comparison_df
    .filter(
        F.col("tgt.policy_id").isNotNull()
        & ~prod_change_condition
    )
    .select("src.*")
)

prod_insert_count = prod_insert_df.count()
prod_update_count = prod_update_df.count()
prod_noop_count = prod_noop_df.count()

prod_total_reject_count = (
    prod_reject_count + prod_duplicate_count
)

print("Production Policy classification completed.")
print("----------------------------------------")
print(f"Incremental records : {prod_data_incremental_count}")
print(f"Valid records       : {prod_valid_count}")
print(f"Duplicate versions  : {prod_duplicate_count}")
print(f"Silver-ready        : {prod_silver_ready_count}")
print(f"INSERT candidates   : {prod_insert_count}")
print(f"UPDATE candidates   : {prod_update_count}")
print(f"NO-OP records       : {prod_noop_count}")
print(f"Total rejects       : {prod_total_reject_count}")

assert (
    prod_insert_count
    + prod_update_count
    + prod_noop_count
    == prod_silver_ready_count
), "Production Policy classification reconciliation failed."

display(prod_silver_ready_policy_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 94, Finished, Available, Finished, False)

Production Policy classification completed.
----------------------------------------
Incremental records : 1
Valid records       : 1
Duplicate versions  : 0
Silver-ready        : 1
INSERT candidates   : 1
UPDATE candidates   : 0
NO-OP records       : 0
Total rejects       : 0


SynapseWidget(Synapse.DataFrame, 9ae10c1b-e9e1-419c-a351-7a3175a5f5ab)

## Step 24 — MERGE Production Policy Changes into Silver and Write Audit

The production pipeline has completed validation, deduplication,
transformation, and change classification.

The classified records are now applied to the Silver Policy Delta table.

### MERGE behavior

Using `policy_id` as the business key:

- New Policies are **INSERTED**
- Existing changed Policies are **UPDATED**
- Unchanged Policies are not written again

After the MERGE, the pipeline verifies that the target row count changed
exactly as expected.

A SUCCESS audit record is then written for the current production batch.

### Important

The control watermark is **not advanced in this step**.

The watermark will move only after the Silver MERGE and audit verification
have completed successfully.

### Expected result for this run

- INSERT: **1**
- UPDATE: **0**
- Silver rows before: **750**
- Silver rows after: **751**
- Audit status: **SUCCESS**

In [93]:
# ---------------------------------------------------------
# STEP 24 - DELTA MERGE + PRODUCTION AUDIT
# ---------------------------------------------------------

from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, LongType
)

# ---------------------------------------------------------
# CAPTURE TARGET STATE BEFORE MERGE
# ---------------------------------------------------------

prod_silver_before_count = (
    spark.table(PROD_CONFIG_TARGET_TABLE).count()
)

prod_test_before_count = (
    spark.table(PROD_CONFIG_TARGET_TABLE)
    .filter(F.col("policy_id") == PROD_TEST_POLICY_ID)
    .count()
)

assert prod_test_before_count == 0, (
    f"{PROD_TEST_POLICY_ID} unexpectedly already exists in Silver."
)


# ---------------------------------------------------------
# BUILD MERGE SOURCE
# ---------------------------------------------------------

prod_merge_source_df = (
    prod_insert_df
    .unionByName(
        prod_update_df,
        allowMissingColumns=True
    )
)

prod_merge_source_count = prod_merge_source_df.count()

assert (
    prod_merge_source_count
    == prod_insert_count + prod_update_count
), "Production MERGE source reconciliation failed."


print("Preparing production Policy MERGE.")
print("----------------------------------------")
print(f"Silver rows before : {prod_silver_before_count}")
print(f"INSERT records     : {prod_insert_count}")
print(f"UPDATE records     : {prod_update_count}")
print(f"MERGE source       : {prod_merge_source_count}")


# ---------------------------------------------------------
# EXECUTE DELTA MERGE
# ---------------------------------------------------------

if prod_merge_source_count > 0:

    prod_silver_delta = DeltaTable.forName(
        spark,
        PROD_CONFIG_TARGET_TABLE
    )

    (
        prod_silver_delta.alias("tgt")
        .merge(
            prod_merge_source_df.alias("src"),
            "tgt.policy_id = src.policy_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print()
    print("Production Policy Delta MERGE completed.")

else:

    print()
    print("No INSERT or UPDATE records detected.")
    print("Production Delta MERGE skipped.")


# ---------------------------------------------------------
# VERIFY SILVER
# ---------------------------------------------------------

prod_silver_after_df = spark.table(
    PROD_CONFIG_TARGET_TABLE
)

prod_silver_after_count = prod_silver_after_df.count()

expected_silver_after_count = (
    prod_silver_before_count + prod_insert_count
)

assert prod_silver_after_count == expected_silver_after_count, (
    "Silver row-count reconciliation failed. "
    f"Expected {expected_silver_after_count}, "
    f"found {prod_silver_after_count}."
)

prod_test_after_df = (
    prod_silver_after_df
    .filter(F.col("policy_id") == PROD_TEST_POLICY_ID)
)

prod_test_after_count = prod_test_after_df.count()

assert prod_test_after_count == 1, (
    f"Expected exactly one {PROD_TEST_POLICY_ID} "
    "record in Silver after MERGE."
)


print()
print("Production Silver MERGE verification PASSED.")
print("----------------------------------------")
print(f"Silver rows before : {prod_silver_before_count}")
print(f"Silver rows after  : {prod_silver_after_count}")
print(
    f"Net row increase   : "
    f"{prod_silver_after_count - prod_silver_before_count}"
)
print(f"Test Policy rows   : {prod_test_after_count}")

display(
    prod_test_after_df.select(
        "policy_id",
        "customer_id",
        "product_type",
        "annual_premium",
        "policy_status",
        "last_updated",
        "_silver_processed_ts"
    )
)


# ---------------------------------------------------------
# WRITE SUCCESS AUDIT
# ---------------------------------------------------------

PROD_RUN_END_TIME = datetime.now()

prod_audit_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("source_count", LongType(), False),
    StructField("insert_count", LongType(), False),
    StructField("update_count", LongType(), False),
    StructField("reject_count", LongType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

prod_audit_record = [(
    PROD_BATCH_ID,
    PROD_PIPELINE_NAME,
    PROD_SOURCE_NAME,
    PROD_RUN_START_TIME,
    PROD_RUN_END_TIME,
    int(prod_data_incremental_count),
    int(prod_insert_count),
    int(prod_update_count),
    int(prod_total_reject_count),
    "SUCCESS",
    None
)]

prod_audit_df = spark.createDataFrame(
    prod_audit_record,
    schema=prod_audit_schema
)

(
    prod_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(PROD_AUDIT_TABLE)
)


# Verify this batch's audit
prod_audit_check_df = (
    spark.table(PROD_AUDIT_TABLE)
    .filter(F.col("batch_id") == PROD_BATCH_ID)
)

prod_audit_check_count = prod_audit_check_df.count()

assert prod_audit_check_count == 1, (
    f"Expected exactly one audit record for batch "
    f"{PROD_BATCH_ID}, found {prod_audit_check_count}."
)

assert (
    prod_audit_check_df.first()["status"] == "SUCCESS"
), "Production audit status is not SUCCESS."


print()
print("Production SUCCESS audit written.")
print("----------------------------------------")
print(f"Batch ID      : {PROD_BATCH_ID}")
print(f"Source count  : {prod_data_incremental_count}")
print(f"Insert count  : {prod_insert_count}")
print(f"Update count  : {prod_update_count}")
print(f"Reject count  : {prod_total_reject_count}")
print("Status        : SUCCESS")

display(prod_audit_check_df)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 95, Finished, Available, Finished, False)

Preparing production Policy MERGE.
----------------------------------------
Silver rows before : 750
INSERT records     : 1
UPDATE records     : 0
MERGE source       : 1

Production Policy Delta MERGE completed.

Production Silver MERGE verification PASSED.
----------------------------------------
Silver rows before : 750
Silver rows after  : 751
Net row increase   : 1
Test Policy rows   : 1


SynapseWidget(Synapse.DataFrame, 32050d95-64ae-4912-bb56-dda43f1257d1)


Production SUCCESS audit written.
----------------------------------------
Batch ID      : 3be3bb42-04af-4ea5-ae98-7ec5812db192
Source count  : 1
Insert count  : 1
Update count  : 0
Reject count  : 0
Status        : SUCCESS


SynapseWidget(Synapse.DataFrame, 6f1cbce2-9e23-4e16-b5d2-1ff53bc2c171)

## Step 25 — Advance Watermark and Verify Idempotent Restart

The production Policy batch has successfully completed:

- Incremental detection
- Validation and reject handling
- Deduplication
- Silver transformation
- INSERT / UPDATE classification
- Delta MERGE
- Silver reconciliation
- SUCCESS audit

The control watermark can now safely advance to the maximum successfully
processed `last_updated` timestamp.

After advancing the watermark, the pipeline performs a restart test.

### Expected Result

- Previous watermark: **2027-12-09**
- New watermark: **2027-12-10**
- Silver Policy rows: **751**
- `POL999901` occurs exactly once in Silver
- Restart incremental records: **0**

This confirms that the Policy incremental pipeline is restart-safe and
idempotent.

In [94]:

# ---------------------------------------------------------
# STEP 25 - ADVANCE WATERMARK + RESTART VERIFICATION
# ---------------------------------------------------------

# Maximum successfully processed watermark
PROD_NEW_WATERMARK = (
    prod_silver_ready_policy_df
    .agg(
        F.max(F.col("last_updated"))
        .alias("new_watermark")
    )
    .first()["new_watermark"]
)

assert PROD_NEW_WATERMARK is not None, (
    "New production watermark cannot be NULL."
)

assert PROD_NEW_WATERMARK > PROD_DATA_LAST_WATERMARK, (
    "New production watermark must be greater than "
    "the previous watermark."
)

print("Production watermark ready for advancement.")
print("----------------------------------------")
print(f"Previous watermark : {PROD_DATA_LAST_WATERMARK}")
print(f"New watermark      : {PROD_NEW_WATERMARK}")


# ---------------------------------------------------------
# UPDATE CONTROL TABLE
# ---------------------------------------------------------

prod_control_delta = DeltaTable.forName(
    spark,
    PROD_CONTROL_TABLE
)

(
    prod_control_delta.alias("tgt")
    .update(
        condition=(
            (F.col("source_name") == PROD_SOURCE_NAME)
            & (F.col("is_active") == True)
        ),
        set={
            "last_watermark": F.lit(PROD_NEW_WATERMARK),
            "_updated_ts": F.current_timestamp()
        }
    )
)


# ---------------------------------------------------------
# VERIFY CONTROL TABLE
# ---------------------------------------------------------

prod_final_control_df = (
    spark.table(PROD_CONTROL_TABLE)
    .filter(
        (F.col("source_name") == PROD_SOURCE_NAME)
        & (F.col("is_active") == True)
    )
)

prod_final_control = prod_final_control_df.first()

assert (
    prod_final_control["last_watermark"]
    == PROD_NEW_WATERMARK
), "Production watermark update verification failed."

print()
print("Production watermark advancement PASSED.")
print("----------------------------------------")
print(
    f"Stored watermark : "
    f"{prod_final_control['last_watermark']}"
)

display(prod_final_control_df)


# ---------------------------------------------------------
# IDEMPOTENT RESTART TEST
# ---------------------------------------------------------

prod_restart_df = (
    spark.table(PROD_CONFIG_SOURCE_TABLE)

    .withColumn(
        "_watermark_ts",
        F.to_timestamp(
            F.col(PROD_DATA_WATERMARK_COLUMN)
        )
    )

    .filter(
        F.col("_watermark_ts")
        > F.lit(PROD_NEW_WATERMARK)
    )
)

prod_restart_count = prod_restart_df.count()


# ---------------------------------------------------------
# FINAL SILVER VERIFICATION
# ---------------------------------------------------------

prod_final_silver_df = spark.table(
    PROD_CONFIG_TARGET_TABLE
)

prod_final_silver_count = prod_final_silver_df.count()

prod_final_test_df = (
    prod_final_silver_df
    .filter(F.col("policy_id") == PROD_TEST_POLICY_ID)
)

prod_final_test_count = prod_final_test_df.count()


assert prod_restart_count == 0, (
    f"Expected 0 incremental records after restart, "
    f"found {prod_restart_count}."
)

assert prod_final_test_count == 1, (
    f"Expected exactly one {PROD_TEST_POLICY_ID} "
    "in Silver."
)


print()
print("========================================")
print(" POLICY PRODUCTION PIPELINE PASSED")
print("========================================")
print(f"Bronze rows          : {prod_data_bronze_count}")
print(f"Silver rows          : {prod_final_silver_count}")
print(f"Processed INSERTS    : {prod_insert_count}")
print(f"Processed UPDATES    : {prod_update_count}")
print(f"Rejected records     : {prod_total_reject_count}")
print(f"Final watermark      : {PROD_NEW_WATERMARK}")
print(f"Restart records      : {prod_restart_count}")
print(f"{PROD_TEST_POLICY_ID} rows : {prod_final_test_count}")

display(
    prod_final_test_df.select(
        "policy_id",
        "customer_id",
        "product_type",
        "annual_premium",
        "policy_status",
        "last_updated",
        "_silver_processed_ts"
    )
)

StatementMeta(, 26494065-8910-49f6-81a1-a58436cfe5ef, 96, Finished, Available, Finished, False)

Production watermark ready for advancement.
----------------------------------------
Previous watermark : 2027-12-09 00:00:00
New watermark      : 2027-12-10 00:00:00

Production watermark advancement PASSED.
----------------------------------------
Stored watermark : 2027-12-10 00:00:00


SynapseWidget(Synapse.DataFrame, 7a59449e-3ba6-405d-ae81-4b0f02355fc3)


 POLICY PRODUCTION PIPELINE PASSED
Bronze rows          : 753
Silver rows          : 751
Processed INSERTS    : 1
Processed UPDATES    : 0
Rejected records     : 0
Final watermark      : 2027-12-10 00:00:00
Restart records      : 0
POL999901 rows : 1


SynapseWidget(Synapse.DataFrame, 637aa904-c0c5-4c64-b7f8-7fd63871c1c7)